# BabyLM Full-Vocabulary Soft Target



In [ ]:

# 0. Install dependencies

# In Colab, run this once.
!pip -q install "transformers==4.44.2" "datasets==2.21.0" accelerate evaluate tqdm pandas matplotlib scipy safetensors


In [ ]:
# 1. Imports
import os
import math
import json
import random
import time
import gc
import shutil
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Optional, Tuple, List, Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    GPT2Config,
    GPT2LMHeadModel,
    get_cosine_schedule_with_warmup,
)


In [ ]:
# 2. Google Drive mount and root directory

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_ROOT = Path("/content/drive/MyDrive/babylm_full_vocab_soft_target_runs")
except Exception:
    DEFAULT_ROOT = Path("./babylm_full_vocab_soft_target_runs")

DEFAULT_ROOT.mkdir(parents=True, exist_ok=True)
print("Project root:", DEFAULT_ROOT)


Project root: babylm_full_vocab_soft_target_runs


In [ ]:
# 3. Global experiment configuration


# Final stable run for the thesis/paper deadline.
# Core 5 functions × seed 42 first.
# After seed 42 works, change SEEDS to [42, 43, 44] and keep the same EXPERIMENT_NAME.
RUN_PROFILE = "paper"

RUN_FULL_EXPERIMENT = True
EXPERIMENT_STAGE = "custom"

# Fresh folder: do not mix with previous bad-warmup/full-eval runs.
EXPERIMENT_NAME = "v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal"

DATA_SOURCE = "hf_babylm"
HF_BABYLM_DATASET = "BabyLM-community/BabyLM-2026-Strict-Small"
LOCAL_TRAIN_TXT = "/content/train.txt"
LOCAL_VALID_TXT = None

# First run seed 42 only
SEEDS = [42,43,44]

RESUME_IF_AVAILABLE = True
SKIP_FINISHED = True
CHECKPOINT_SELECTION_POLICY = "best_val_loss"



RUN_TAU_DIAGNOSTICS = True
RUN_ENTROPY_BUCKET_EVAL = True
RUN_EMBEDDING_DISTANCE_DIAG = True

# Static full-vocab τ grid.
FULL_VOCAB_STATIC_TAU_GRID = [0.01, 0.02, 0.03, 0.05, 0.055, 0.07]

# Best τ found from full-vocab inspection!
BEST_FULL_VOCAB_STATIC_TAU = 0.055

DYNAMIC_TAU_MIN = 0.01
DYNAMIC_TAU_MID = 0.03
DYNAMIC_TAU_MAX = 0.055
DYNAMIC_HARD_GATE = 0.50
MIX_ALPHA = 0.90  # 90% hard CE + 10% soft CE

def tau_to_tag(tau: float) -> str:
    return f"{tau:.3f}".rstrip("0").rstrip(".").replace(".", "p")

def alpha_to_tag(alpha: float) -> str:
    return f"{alpha:.2f}".rstrip("0").rstrip(".").replace(".", "p")


In [ ]:

# 4. Config dataclasses


@dataclass
class ModelConfigSmall:
    n_layer: int
    n_head: int
    n_embd: int
    block_size: int

@dataclass
class TrainConfig:
    run_profile: str
    batch_size: int
    grad_accum_steps: int
    learning_rate: float
    weight_decay: float
    max_steps: int
    warmup_steps: int
    eval_every: int
    save_every: int
    num_workers: int
    use_amp: bool
    max_grad_norm: float
    eval_max_batches: Optional[int]
    entropy_stats_max_batches: Optional[int]
    raw_train_row_limit: Optional[int]
    raw_valid_row_limit: Optional[int]
    train_block_limit: Optional[int]
    valid_block_limit: Optional[int]
    early_stopping_patience: Optional[int]
    early_stopping_min_steps: int
    early_stopping_min_delta: float
    full_vocab_chunk_size: int
    tau_grid_size: int

@dataclass
class ConditionSpec:
    name: str
    kind: str  # "hard", "static", "piecewise", "sigmoid", "sigmoid_hard_gate", "soft_target_entropy"
    static_tau: Optional[float] = None
    tau_min: float = DYNAMIC_TAU_MIN
    tau_mid: float = DYNAMIC_TAU_MID
    tau_max: float = DYNAMIC_TAU_MAX
    sigmoid_k: float = 8.0
    sigmoid_center: float = 0.5
    hard_gate: float = DYNAMIC_HARD_GATE
    mix_alpha: Optional[float] = None
    shuffled_tau: bool = False
    label_smoothing_epsilon: Optional[float] = None

if RUN_PROFILE == "smoke":
    MODEL_CFG = ModelConfigSmall(n_layer=2, n_head=2, n_embd=128, block_size=64)
    TRAIN_CFG = TrainConfig(
        run_profile=RUN_PROFILE,
        batch_size=4,
        grad_accum_steps=2,
        learning_rate=5e-4,
        weight_decay=0.01,
        max_steps=40,
        warmup_steps=5,
        eval_every=10,
        save_every=20,
        num_workers=0,
        use_amp=True,
        max_grad_norm=1.0,
        eval_max_batches=5,
        entropy_stats_max_batches=5,
        raw_train_row_limit=2000,
        raw_valid_row_limit=300,
        train_block_limit=400,
        valid_block_limit=80,
        early_stopping_patience=None,
        early_stopping_min_steps=0,
        early_stopping_min_delta=1e-4,
        full_vocab_chunk_size=32,
        tau_grid_size=8,
    )

elif RUN_PROFILE == "pilot":
    MODEL_CFG = ModelConfigSmall(n_layer=6, n_head=6, n_embd=384, block_size=256)
    TRAIN_CFG = TrainConfig(
        run_profile=RUN_PROFILE,
        batch_size=8,
        grad_accum_steps=4,
        learning_rate=5e-4,
        weight_decay=0.01,
        max_steps=2000,
        warmup_steps=100,
        eval_every=250,
        save_every=500,
        num_workers=0,
        use_amp=True,
        max_grad_norm=1.0,
        eval_max_batches=25,
        entropy_stats_max_batches=50,
        raw_train_row_limit=20000,
        raw_valid_row_limit=2000,
        train_block_limit=6000,
        valid_block_limit=800,
        early_stopping_patience=5,
        early_stopping_min_steps=1000,
        early_stopping_min_delta=1e-4,
        full_vocab_chunk_size=64,
        tau_grid_size=12,
    )

elif RUN_PROFILE == "paper":
    # Stable paper-scale setup.
    # Model remains compact (128d) to keep full-vocabulary soft targets feasible.
    MODEL_CFG = ModelConfigSmall(n_layer=8, n_head=8, n_embd=128, block_size=256)
    TRAIN_CFG = TrainConfig(
        run_profile=RUN_PROFILE,
        batch_size=8,
        grad_accum_steps=8,
        learning_rate=5e-4,
        weight_decay=0.01,
        max_steps=20000,
        # IMPORTANT:
        # scheduler.step() happens once per optimizer update.
        # With grad_accum_steps=8, warmup_steps=125 means about 1000 displayed training steps.
        # Do NOT set this to 1000 here, or the displayed warmup becomes about 8000 steps.
        warmup_steps=125,

        # Each condition is also full-evaluated once at the end on the selected checkpoint.
        eval_every=1000,
        save_every=1000,
        num_workers=2,
        use_amp=True,
        max_grad_norm=1.0,
        eval_max_batches=25,
        entropy_stats_max_batches=None,
        raw_train_row_limit=None,
        raw_valid_row_limit=None,
        train_block_limit=None,
        valid_block_limit=None,
        early_stopping_patience=5,
        early_stopping_min_steps=2000,
        early_stopping_min_delta=1e-4,
        full_vocab_chunk_size=64,
        tau_grid_size=16,
    )

else:
    raise ValueError(f"Unknown RUN_PROFILE: {RUN_PROFILE}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Model config:", MODEL_CFG)
print("Train config:", TRAIN_CFG)


Device: cuda
Model config: ModelConfigSmall(n_layer=8, n_head=8, n_embd=128, block_size=256)
Train config: TrainConfig(run_profile='paper', batch_size=8, grad_accum_steps=8, learning_rate=0.0005, weight_decay=0.01, max_steps=20000, warmup_steps=125, eval_every=1000, save_every=1000, num_workers=2, use_amp=True, max_grad_norm=1.0, eval_max_batches=25, entropy_stats_max_batches=None, raw_train_row_limit=None, raw_valid_row_limit=None, train_block_limit=None, valid_block_limit=None, early_stopping_patience=5, early_stopping_min_steps=2000, early_stopping_min_delta=0.0001, full_vocab_chunk_size=64, tau_grid_size=16)


In [ ]:
# 5. Build condition specs
CONDITION_ALIASES = {
    "hard_ce": "hard_ce",
    "full_vocab_static_tau_0p055": "static_soft_pure",
    "full_vocab_static_tau_0p055_mix_alpha_0p9": "static_soft_mixed",
    "full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9": "sigmoid_hard_gate",
    "full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9_shuffled_tau": "sigmoid_hard_gate_shuffled",
    "full_vocab_piecewise_tau_max_0p055_mix_alpha_0p9": "piecewise_entropy",
}

def build_condition_specs(stage: str) -> Dict[str, ConditionSpec]:
    specs: Dict[str, ConditionSpec] = {}

    # Always include the hard-target baseline.
    specs["hard_ce"] = ConditionSpec(name="hard_ce", kind="hard")

    if stage == "full_vocab_static_grid":
        for tau in FULL_VOCAB_STATIC_TAU_GRID:
            name = f"full_vocab_static_tau_{tau_to_tag(tau)}"
            specs[name] = ConditionSpec(name=name, kind="static", static_tau=tau)

    elif stage == "full_vocab_rescue":
        # Pilot/rescue stage.
        # Kept for reference
        best_tau = BEST_FULL_VOCAB_STATIC_TAU
        best_tag = tau_to_tag(best_tau)
        alpha_tag = alpha_to_tag(MIX_ALPHA)
        tau_max_tag = tau_to_tag(DYNAMIC_TAU_MAX)
        gate_tag = tau_to_tag(DYNAMIC_HARD_GATE)

        specs[f"full_vocab_static_tau_{best_tag}"] = ConditionSpec(
            name=f"full_vocab_static_tau_{best_tag}",
            kind="static",
            static_tau=best_tau,
        )

        specs[f"full_vocab_static_tau_{best_tag}_mix_alpha_{alpha_tag}"] = ConditionSpec(
            name=f"full_vocab_static_tau_{best_tag}_mix_alpha_{alpha_tag}",
            kind="static",
            static_tau=best_tau,
            mix_alpha=MIX_ALPHA,
        )

        specs[f"full_vocab_soft_target_entropy_tau_max_{tau_max_tag}_mix_alpha_{alpha_tag}"] = ConditionSpec(
            name=f"full_vocab_soft_target_entropy_tau_max_{tau_max_tag}_mix_alpha_{alpha_tag}",
            kind="soft_target_entropy",
            tau_min=DYNAMIC_TAU_MIN,
            tau_mid=DYNAMIC_TAU_MID,
            tau_max=DYNAMIC_TAU_MAX,
            mix_alpha=MIX_ALPHA,
        )

        specs[f"full_vocab_sigmoid_hard_gate_tau_max_{tau_max_tag}_gate_{gate_tag}_mix_alpha_{alpha_tag}"] = ConditionSpec(
            name=f"full_vocab_sigmoid_hard_gate_tau_max_{tau_max_tag}_gate_{gate_tag}_mix_alpha_{alpha_tag}",
            kind="sigmoid_hard_gate",
            tau_min=DYNAMIC_TAU_MIN,
            tau_mid=DYNAMIC_TAU_MID,
            tau_max=DYNAMIC_TAU_MAX,
            hard_gate=DYNAMIC_HARD_GATE,
            mix_alpha=MIX_ALPHA,
        )

        specs[f"full_vocab_sigmoid_hard_gate_tau_max_{tau_max_tag}_gate_{gate_tag}_mix_alpha_{alpha_tag}_shuffled_tau"] = ConditionSpec(
            name=f"full_vocab_sigmoid_hard_gate_tau_max_{tau_max_tag}_gate_{gate_tag}_mix_alpha_{alpha_tag}_shuffled_tau",
            kind="sigmoid_hard_gate",
            tau_min=DYNAMIC_TAU_MIN,
            tau_mid=DYNAMIC_TAU_MID,
            tau_max=DYNAMIC_TAU_MAX,
            hard_gate=DYNAMIC_HARD_GATE,
            mix_alpha=MIX_ALPHA,
            shuffled_tau=True,
        )

        specs[f"full_vocab_piecewise_tau_max_{tau_max_tag}_mix_alpha_{alpha_tag}"] = ConditionSpec(
            name=f"full_vocab_piecewise_tau_max_{tau_max_tag}_mix_alpha_{alpha_tag}",
            kind="piecewise",
            tau_min=DYNAMIC_TAU_MIN,
            tau_mid=DYNAMIC_TAU_MID,
            tau_max=DYNAMIC_TAU_MAX,
            mix_alpha=MIX_ALPHA,
        )

    elif stage == "custom":
        # Final core 5 functions.
        # hard_ce is already included above.
        best_tau = BEST_FULL_VOCAB_STATIC_TAU
        best_tag = tau_to_tag(best_tau)
        alpha_tag = alpha_to_tag(MIX_ALPHA)
        tau_max_tag = tau_to_tag(DYNAMIC_TAU_MAX)
        gate_tag = tau_to_tag(DYNAMIC_HARD_GATE)

        # 1. Best pure full-vocabulary static soft target.
        specs[f"full_vocab_static_tau_{best_tag}"] = ConditionSpec(
            name=f"full_vocab_static_tau_{best_tag}",
            kind="static",
            static_tau=best_tau,
        )

        # 2. Main dynamic method:
        # entropy-conditioned sigmoid hard gate with mixed loss.
        specs[f"full_vocab_sigmoid_hard_gate_tau_max_{tau_max_tag}_gate_{gate_tag}_mix_alpha_{alpha_tag}"] = ConditionSpec(
            name=f"full_vocab_sigmoid_hard_gate_tau_max_{tau_max_tag}_gate_{gate_tag}_mix_alpha_{alpha_tag}",
            kind="sigmoid_hard_gate",
            tau_min=DYNAMIC_TAU_MIN,
            tau_mid=DYNAMIC_TAU_MID,
            tau_max=DYNAMIC_TAU_MAX,
            hard_gate=DYNAMIC_HARD_GATE,
            mix_alpha=MIX_ALPHA,
        )

        # 3. Shuffled dynamic control:
        # same τ distribution as dynamic, but entropy-to-τ alignment is destroyed.
        specs[f"full_vocab_sigmoid_hard_gate_tau_max_{tau_max_tag}_gate_{gate_tag}_mix_alpha_{alpha_tag}_shuffled_tau"] = ConditionSpec(
            name=f"full_vocab_sigmoid_hard_gate_tau_max_{tau_max_tag}_gate_{gate_tag}_mix_alpha_{alpha_tag}_shuffled_tau",
            kind="sigmoid_hard_gate",
            tau_min=DYNAMIC_TAU_MIN,
            tau_mid=DYNAMIC_TAU_MID,
            tau_max=DYNAMIC_TAU_MAX,
            hard_gate=DYNAMIC_HARD_GATE,
            mix_alpha=MIX_ALPHA,
            shuffled_tau=True,
        )

        # 4. Piecewise entropy dynamic:
        # best remaining already-tested dynamic variant after the core dynamic method.
        specs[f"full_vocab_piecewise_tau_max_{tau_max_tag}_mix_alpha_{alpha_tag}"] = ConditionSpec(
            name=f"full_vocab_piecewise_tau_max_{tau_max_tag}_mix_alpha_{alpha_tag}",
            kind="piecewise",
            tau_min=DYNAMIC_TAU_MIN,
            tau_mid=DYNAMIC_TAU_MID,
            tau_max=DYNAMIC_TAU_MAX,
            mix_alpha=MIX_ALPHA,
        )

    else:
        raise ValueError(f"Unknown EXPERIMENT_STAGE: {stage}")

    return specs

CONDITION_SPECS = build_condition_specs(EXPERIMENT_STAGE)
CONDITIONS = list(CONDITION_SPECS.keys())

print("Conditions:")
for c in CONDITIONS:
    print(" -", c, "=>", CONDITION_ALIASES.get(c, c), CONDITION_SPECS[c])


Conditions:
 - hard_ce => hard_ce ConditionSpec(name='hard_ce', kind='hard', static_tau=None, tau_min=0.01, tau_mid=0.03, tau_max=0.055, sigmoid_k=8.0, sigmoid_center=0.5, hard_gate=0.5, mix_alpha=None, shuffled_tau=False, label_smoothing_epsilon=None)
 - full_vocab_static_tau_0p055 => static_soft_pure ConditionSpec(name='full_vocab_static_tau_0p055', kind='static', static_tau=0.055, tau_min=0.01, tau_mid=0.03, tau_max=0.055, sigmoid_k=8.0, sigmoid_center=0.5, hard_gate=0.5, mix_alpha=None, shuffled_tau=False, label_smoothing_epsilon=None)
 - full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9 => sigmoid_hard_gate ConditionSpec(name='full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9', kind='sigmoid_hard_gate', static_tau=None, tau_min=0.01, tau_mid=0.03, tau_max=0.055, sigmoid_k=8.0, sigmoid_center=0.5, hard_gate=0.5, mix_alpha=0.9, shuffled_tau=False, label_smoothing_epsilon=None)
 - full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9_shuffl

In [ ]:
# 6. Reproducibility and torch helpers

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def maybe_autocast():
    if device.type == "cuda" and TRAIN_CFG.use_amp:
        return torch.cuda.amp.autocast()
    from contextlib import nullcontext
    return nullcontext()

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# 7. Directory helpers and checkpoint validity

RUN_ROOT = DEFAULT_ROOT / EXPERIMENT_NAME
CKPT_ROOT = RUN_ROOT / "checkpoints"
LOG_DIR = RUN_ROOT / "logs"
EVAL_DIR = RUN_ROOT / "evals"
DIAG_DIR = RUN_ROOT / "diagnostics"
CACHE_DIR = RUN_ROOT / "cache"

for d in [CKPT_ROOT, LOG_DIR, EVAL_DIR, DIAG_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def condition_root(seed: int, condition: str) -> Path:
    return CKPT_ROOT / f"seed_{seed}" / condition

def step_checkpoint_dir(seed: int, condition: str, step: int) -> Path:
    return condition_root(seed, condition) / f"step_{step}"

def best_checkpoint_dir(seed: int, condition: str) -> Path:
    return condition_root(seed, condition) / "best_by_val_loss"

def final_checkpoint_dir(seed: int, condition: str) -> Path:
    return condition_root(seed, condition) / "final"

def log_path(seed: int, condition: str) -> Path:
    return LOG_DIR / f"train_log_seed_{seed}_{condition}.csv"

def summary_path(seed: int, condition: str) -> Path:
    return LOG_DIR / f"summary_seed_{seed}_{condition}.json"

def completion_marker_path(seed: int, condition: str) -> Path:
    return condition_root(seed, condition) / "COMPLETED.json"

def checkpoint_has_model_weights(path: Path) -> bool:
    path = Path(path)
    if not path.exists() or not path.is_dir():
        return False
    has_config = (path / "config.json").exists()
    weight_names = [
        "model.safetensors",
        "pytorch_model.bin",
        "model.safetensors.index.json",
        "pytorch_model.bin.index.json",
    ]
    has_weights = any((path / name).exists() for name in weight_names)
    return bool(has_config and has_weights)

def parse_step_from_path(path: Path) -> Optional[int]:
    m = re.search(r"step_(\d+)$", str(path))
    return int(m.group(1)) if m else None

def list_condition_checkpoints(seed: int, condition: str) -> List[Path]:
    root = condition_root(seed, condition)
    if not root.exists():
        return []
    ckpts = [
        p for p in root.glob("step_*")
        if p.is_dir() and parse_step_from_path(p) is not None and checkpoint_has_model_weights(p)
    ]
    return sorted(ckpts, key=lambda p: parse_step_from_path(p))

def latest_checkpoint_dir(seed: int, condition: str) -> Optional[Path]:
    ckpts = list_condition_checkpoints(seed, condition)
    return ckpts[-1] if ckpts else None

def selected_checkpoint_dir(seed: int, condition: str) -> Optional[Path]:
    best_dir = best_checkpoint_dir(seed, condition)
    if CHECKPOINT_SELECTION_POLICY == "best_val_loss" and checkpoint_has_model_weights(best_dir):
        return best_dir
    final_dir = final_checkpoint_dir(seed, condition)
    if checkpoint_has_model_weights(final_dir):
        return final_dir
    latest = latest_checkpoint_dir(seed, condition)
    if latest is not None and checkpoint_has_model_weights(latest):
        return latest
    return None

def save_json(obj: Dict[str, Any], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r") as f:
        return json.load(f)

print("Run root:", RUN_ROOT)


# Save run configuration and condition table.
condition_table = pd.DataFrame([
    {
        "condition": name,
        "short_name": CONDITION_ALIASES.get(name, name),
        **asdict(spec),
    }
    for name, spec in CONDITION_SPECS.items()
])
condition_table_path = LOG_DIR / "condition_table.csv"
condition_table.to_csv(condition_table_path, index=False)

run_manifest = {
    "experiment_name": EXPERIMENT_NAME,
    "run_profile": RUN_PROFILE,
    "experiment_stage": EXPERIMENT_STAGE,
    "seeds": SEEDS,
    "model_config": asdict(MODEL_CFG),
    "train_config": asdict(TRAIN_CFG),
    "checkpoint_selection_policy": CHECKPOINT_SELECTION_POLICY,
    "conditions": CONDITIONS,
    "condition_aliases": {c: CONDITION_ALIASES.get(c, c) for c in CONDITIONS},
}
save_json(run_manifest, RUN_ROOT / "run_manifest.json")

print("Saved run manifest:", RUN_ROOT / "run_manifest.json")
print("Saved condition table:", condition_table_path)
display(condition_table[["condition", "short_name", "kind", "static_tau", "tau_min", "tau_mid", "tau_max", "hard_gate", "mix_alpha", "shuffled_tau"]])


Run root: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal
Saved run manifest: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/run_manifest.json
Saved condition table: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/logs/condition_table.csv


,condition,short_name,kind,static_tau,tau_min,tau_mid,tau_max,hard_gate,mix_alpha,shuffled_tau
0,hard_ce,hard_ce,hard,NaN,0.01,0.03,0.055,0.5,NaN,False
1,full_vocab_static_tau_0p055,static_soft_pure,static,0.055,0.01,0.03,0.055,0.5,NaN,False
2,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,sigmoid_hard_gate,sigmoid_hard_gate,NaN,0.01,0.03,0.055,0.5,0.9,False
3,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,sigmoid_hard_gate_shuffled,sigmoid_hard_gate,NaN,0.01,0.03,0.055,0.5,0.9,True
4,full_vocab_piecewise_tau_max_0p055_mix_alpha_0p9,piecewise_entropy,piecewise,NaN,0.01,0.03,0.055,0.5,0.9,False


In [ ]:
# 8. Load tokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

print("vocab size:", len(tokenizer))
print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocab size: 50257
pad token: <|endoftext|> 50256


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
# 9. Load raw dataset

def detect_text_column(ds_split) -> str:
    candidates = ["text", "sentence", "content"]
    for c in candidates:
        if c in ds_split.column_names:
            return c
    # Fall back to first string column.
    for c in ds_split.column_names:
        try:
            val = ds_split[0][c]
            if isinstance(val, str):
                return c
        except Exception:
            pass
    raise ValueError(f"Could not find text column in columns: {ds_split.column_names}")

def load_raw_data() -> DatasetDict:
    if DATA_SOURCE == "hf_babylm":
        raw = load_dataset(HF_BABYLM_DATASET)
        if "train" not in raw:
            raise ValueError("HF dataset needs a train split.")
        if "validation" not in raw:
            split = raw["train"].train_test_split(test_size=0.01, seed=42)
            raw = DatasetDict({"train": split["train"], "validation": split["test"]})
        return raw

    if DATA_SOURCE == "local_txt":
        train_path = Path(LOCAL_TRAIN_TXT)
        if not train_path.exists():
            raise FileNotFoundError(f"Missing LOCAL_TRAIN_TXT: {LOCAL_TRAIN_TXT}")
        train_lines = train_path.read_text(encoding="utf-8").splitlines()
        train_ds = Dataset.from_dict({"text": [x for x in train_lines if x.strip()]})

        if LOCAL_VALID_TXT is not None and Path(LOCAL_VALID_TXT).exists():
            valid_lines = Path(LOCAL_VALID_TXT).read_text(encoding="utf-8").splitlines()
            valid_ds = Dataset.from_dict({"text": [x for x in valid_lines if x.strip()]})
        else:
            split = train_ds.train_test_split(test_size=0.01, seed=42)
            train_ds, valid_ds = split["train"], split["test"]

        return DatasetDict({"train": train_ds, "validation": valid_ds})

    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}")

raw_ds = load_raw_data()

text_col = detect_text_column(raw_ds["train"])
print("Text column:", text_col)
print(raw_ds)


Generating train split:   0%|          | 0/1104106 [00:00<?, ? examples/s]

Text column: text
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1093064
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 11042
    })
})


In [ ]:
# 10. Tokenize and block dataset


def maybe_select(ds, limit):
    if limit is None:
        return ds
    return ds.select(range(min(limit, len(ds))))

raw_train = maybe_select(raw_ds["train"], TRAIN_CFG.raw_train_row_limit)
raw_valid = maybe_select(raw_ds["validation"], TRAIN_CFG.raw_valid_row_limit)

def tokenize_function(examples):
    texts = [str(x) for x in examples[text_col]]
    return tokenizer(texts, add_special_tokens=False, return_attention_mask=False)

tokenized_train_raw = raw_train.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_train.column_names,
    desc="Tokenizing train",
)
tokenized_valid_raw = raw_valid.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_valid.column_names,
    desc="Tokenizing validation",
)

def group_texts(examples):
    # Concatenate then split into fixed blocks.
    concatenated = []
    for ids in examples["input_ids"]:
        concatenated.extend(ids + [tokenizer.eos_token_id])

    block_size = MODEL_CFG.block_size
    total_length = (len(concatenated) // block_size) * block_size
    concatenated = concatenated[:total_length]

    if total_length == 0:
        return {"input_ids": []}

    blocks = [
        concatenated[i : i + block_size]
        for i in range(0, total_length, block_size)
    ]
    return {"input_ids": blocks}

tokenized_train = tokenized_train_raw.map(
    group_texts,
    batched=True,
    remove_columns=[c for c in tokenized_train_raw.column_names if c != "input_ids"],
    desc="Grouping train",
)
tokenized_valid = tokenized_valid_raw.map(
    group_texts,
    batched=True,
    remove_columns=[c for c in tokenized_valid_raw.column_names if c != "input_ids"],
    desc="Grouping validation",
)

if TRAIN_CFG.train_block_limit is not None:
    tokenized_train = tokenized_train.select(range(min(TRAIN_CFG.train_block_limit, len(tokenized_train))))
if TRAIN_CFG.valid_block_limit is not None:
    tokenized_valid = tokenized_valid.select(range(min(TRAIN_CFG.valid_block_limit, len(tokenized_valid))))

tokenized_ds = DatasetDict({
    "train": tokenized_train,
    "validation": tokenized_valid,
})

print(tokenized_ds)
print("Example block length:", len(tokenized_ds["train"][0]["input_ids"]))


Tokenizing train:   0%|          | 0/1093064 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1028 > 1024). Running this sequence through the model will result in indexing errors


Tokenizing validation:   0%|          | 0/11042 [00:00<?, ? examples/s]

Grouping train:   0%|          | 0/1093064 [00:00<?, ? examples/s]

Grouping validation:   0%|          | 0/11042 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 63277
    })
    validation: Dataset({
        features: ['input_ids'],
        num_rows: 644
    })
})
Example block length: 256


In [ ]:
# 11. Dataloaders


def collate_lm_batch(features):
    input_ids = torch.stack([
        torch.as_tensor(f["input_ids"], dtype=torch.long)
        for f in features
    ], dim=0)
    attention_mask = torch.ones_like(input_ids)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
    }

def make_dataloaders(seed: int, train_shuffle: bool = True):
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        tokenized_ds["train"],
        batch_size=TRAIN_CFG.batch_size,
        shuffle=train_shuffle,
        generator=generator if train_shuffle else None,
        num_workers=TRAIN_CFG.num_workers,
        collate_fn=collate_lm_batch,
        pin_memory=torch.cuda.is_available(),
    )

    valid_loader = DataLoader(
        tokenized_ds["validation"],
        batch_size=TRAIN_CFG.batch_size,
        shuffle=False,
        num_workers=TRAIN_CFG.num_workers,
        collate_fn=collate_lm_batch,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, valid_loader

train_loader_test, valid_loader_test = make_dataloaders(SEEDS[0], train_shuffle=True)
print("train batches:", len(train_loader_test))
print("valid batches:", len(valid_loader_test))
del train_loader_test, valid_loader_test


train batches: 7910
valid batches: 81


In [ ]:

# 12. Build model from scratch


def build_model_from_scratch() -> GPT2LMHeadModel:
    config = GPT2Config(
        vocab_size=len(tokenizer),
        n_positions=MODEL_CFG.block_size,
        n_ctx=MODEL_CFG.block_size,
        n_embd=MODEL_CFG.n_embd,
        n_layer=MODEL_CFG.n_layer,
        n_head=MODEL_CFG.n_head,
        bos_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    model = GPT2LMHeadModel(config)
    return model

m = build_model_from_scratch()
print("Parameters:", sum(p.numel() for p in m.parameters()) / 1e6, "M")
del m
cleanup_cuda()


Parameters: 8.052096 M


In [ ]:
# 13. Entropy and τ functions


def entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    log_probs = F.log_softmax(logits.float(), dim=-1)
    probs = torch.exp(log_probs)
    return -(probs * log_probs).sum(dim=-1)

@torch.no_grad()
def compute_reference_entropy(
    reference_model,
    input_ids: torch.Tensor,
    attention_mask: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    reference_model.eval()
    out = reference_model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits[:, :-1, :].float()
    return entropy_from_logits(logits)

def normalize_entropy(entropy: torch.Tensor, stats: Dict[str, float]) -> torch.Tensor:
    h_low = float(stats["q05"])
    h_high = float(stats["q95"])
    denom = max(h_high - h_low, 1e-8)
    return torch.clamp((entropy - h_low) / denom, 0.0, 1.0)

def tau_piecewise(h_norm: torch.Tensor, spec: ConditionSpec) -> torch.Tensor:
    tau = torch.empty_like(h_norm)
    tau[h_norm < 0.33] = spec.tau_min
    tau[(h_norm >= 0.33) & (h_norm < 0.66)] = spec.tau_mid
    tau[h_norm >= 0.66] = spec.tau_max
    return tau

def tau_sigmoid(h_norm: torch.Tensor, spec: ConditionSpec) -> torch.Tensor:
    sig = torch.sigmoid(spec.sigmoid_k * (h_norm - spec.sigmoid_center))
    return spec.tau_min + sig * (spec.tau_max - spec.tau_min)

def shift_attention_mask(attention_mask: Optional[torch.Tensor], labels_shifted: torch.Tensor) -> torch.Tensor:
    if attention_mask is None:
        return torch.ones_like(labels_shifted, dtype=torch.float)
    return attention_mask[:, 1:].contiguous().float()

def shuffle_flat_valid_values(values: torch.Tensor, valid_mask: torch.Tensor) -> torch.Tensor:
    # Shuffle values only over valid positions, preserving the marginal distribution.
    out = values.clone()
    flat = out.reshape(-1)
    valid_flat = valid_mask.reshape(-1).bool()
    idx = torch.nonzero(valid_flat, as_tuple=False).squeeze(-1)
    if idx.numel() > 1:
        perm = idx[torch.randperm(idx.numel(), device=idx.device)]
        flat[idx] = flat[perm]
    return out

def tau_grid_values(spec: ConditionSpec, n: int) -> torch.Tensor:
    return torch.linspace(
        float(spec.tau_min),
        float(spec.tau_max),
        int(n),
        device=device,
        dtype=torch.float32,
    ).clamp_min(1e-6)


In [ ]:
# 14. Hard CE and full-vocab soft CE losses

def hard_ce_loss_sum_count(
    student_logits: torch.Tensor,
    input_ids: torch.Tensor,
    attention_mask: Optional[torch.Tensor] = None,
    token_mask: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    shift_logits = student_logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    base_mask = shift_attention_mask(attention_mask, shift_labels)

    if token_mask is not None:
        base_mask = base_mask * token_mask.float()

    V = shift_logits.size(-1)
    loss = F.cross_entropy(
        shift_logits.view(-1, V).float(),
        shift_labels.reshape(-1),
        reduction="none",
    ).view_as(shift_labels)

    loss_sum = (loss * base_mask).sum()
    count = base_mask.sum()
    return loss_sum, count

def full_vocab_soft_ce_loss_sum_count(
    student_logits: torch.Tensor,
    input_ids: torch.Tensor,
    attention_mask: Optional[torch.Tensor],
    vocab_emb_norm: torch.Tensor,
    tau: torch.Tensor,
    token_mask: Optional[torch.Tensor] = None,
    chunk_size: Optional[int] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    '''
    Full-vocabulary soft target:
        q_t(v) ∝ exp(cos(e_y, e_v) / tau_t), v ∈ V

    Computes:
        - sum_t sum_v -q_t(v) log p_student(v)
    over valid positions, using chunks over token positions to control memory.
    '''
    if chunk_size is None:
        chunk_size = TRAIN_CFG.full_vocab_chunk_size

    shift_logits = student_logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    base_mask = shift_attention_mask(attention_mask, shift_labels).bool()

    if token_mask is not None:
        base_mask = base_mask & token_mask.bool()

    B, T, V = shift_logits.shape
    logits_flat = shift_logits.reshape(B * T, V)
    labels_flat = shift_labels.reshape(B * T)
    mask_flat = base_mask.reshape(B * T)

    if not torch.is_tensor(tau):
        tau_flat = torch.full((B * T,), float(tau), device=shift_logits.device, dtype=torch.float32)
    else:
        tau_flat = tau.reshape(B * T).to(shift_logits.device).float()

    valid_indices = torch.nonzero(mask_flat, as_tuple=False).squeeze(-1)
    if valid_indices.numel() == 0:
        zero = student_logits.sum() * 0.0
        return zero, torch.tensor(0.0, device=student_logits.device)

    total_loss = student_logits.sum() * 0.0
    total_count = torch.tensor(0.0, device=student_logits.device)

    # Ensure normalized embeddings are on the same device and float32.
    E = vocab_emb_norm.to(shift_logits.device).float()

    for start in range(0, valid_indices.numel(), chunk_size):
        idx = valid_indices[start : start + chunk_size]

        chunk_logits = logits_flat[idx].float()              # [n, V]
        chunk_labels = labels_flat[idx]                      # [n]
        chunk_tau = tau_flat[idx].clamp_min(1e-6)             # [n]

        gold_emb = E[chunk_labels]                           # [n, D]
        target_logits = gold_emb @ E.T                       # [n, V], cosine because E normalized
        target_logits = target_logits / chunk_tau.unsqueeze(-1)

        target_probs = F.softmax(target_logits, dim=-1)      # [n, V]
        student_log_probs = F.log_softmax(chunk_logits, dim=-1)

        loss_vec = -(target_probs * student_log_probs).sum(dim=-1)

        total_loss = total_loss + loss_vec.sum()
        total_count = total_count + loss_vec.numel()

    return total_loss, total_count

@torch.no_grad()
def full_vocab_target_entropy_for_batch(
    labels_shifted: torch.Tensor,
    h_norm: torch.Tensor,
    vocab_emb_norm: torch.Tensor,
    spec: ConditionSpec,
    valid_mask: torch.Tensor,
    chunk_size: Optional[int] = None,
) -> torch.Tensor:
    '''
    Implements:
        context entropy -> desired target entropy -> tau

    This version is full-vocab and batch-local. It grid-searches tau for each
    observed target token in the current batch. It is slower than sigmoid/piecewise.
    '''
    if chunk_size is None:
        chunk_size = max(16, TRAIN_CFG.full_vocab_chunk_size)

    B, T = labels_shifted.shape
    tau_out = torch.full((B, T), float(spec.tau_min), device=labels_shifted.device, dtype=torch.float32)

    labels_flat = labels_shifted.reshape(-1)
    h_flat = h_norm.reshape(-1).float()
    valid_flat = valid_mask.reshape(-1).bool()

    valid_indices = torch.nonzero(valid_flat, as_tuple=False).squeeze(-1)
    if valid_indices.numel() == 0:
        return tau_out

    E = vocab_emb_norm.to(labels_shifted.device).float()
    grid = tau_grid_values(spec, TRAIN_CFG.tau_grid_size)  # [G]

    for start in range(0, valid_indices.numel(), chunk_size):
        idx = valid_indices[start : start + chunk_size]
        y = labels_flat[idx]
        h = h_flat[idx]

        gold_emb = E[y]
        sims = gold_emb @ E.T  # [n, V]

        entropies = []
        for tau_val in grid:
            log_probs = F.log_softmax(sims / tau_val, dim=-1)
            probs = torch.exp(log_probs)
            ent = -(probs * log_probs).sum(dim=-1)
            entropies.append(ent)
        entropies = torch.stack(entropies, dim=0)  # [G, n]

        s_low = entropies[0]
        s_high = entropies[-1]
        desired = s_low + h * (s_high - s_low)

        best_idx = torch.argmin(torch.abs(entropies - desired.unsqueeze(0)), dim=0)
        tau_out.reshape(-1)[idx] = grid[best_idx]

    return tau_out


In [ ]:
# 15. Compute training loss by condition


def get_vocab_emb_norm(reference_model: GPT2LMHeadModel) -> torch.Tensor:
    # Use frozen hard-CE reference embedding matrix for target construction.
    emb = reference_model.transformer.wte.weight.detach().float()
    return F.normalize(emb, dim=-1).contiguous()

def compute_tau_and_masks(
    spec: ConditionSpec,
    reference_model: GPT2LMHeadModel,
    input_ids: torch.Tensor,
    attention_mask: Optional[torch.Tensor],
    entropy_stats: Dict[str, float],
    vocab_emb_norm: torch.Tensor,
) -> Tuple[torch.Tensor, Optional[torch.Tensor], torch.Tensor]:
    '''
    Returns:
        tau: [B, T-1]
        hard_mask: [B, T-1] or None. True means use hard CE for that position in gated non-mixed setup.
        valid_mask: [B, T-1]
    '''
    labels_shifted = input_ids[:, 1:].contiguous()
    valid_mask = shift_attention_mask(attention_mask, labels_shifted).bool()

    ref_entropy = compute_reference_entropy(reference_model, input_ids, attention_mask)
    h_norm = normalize_entropy(ref_entropy, entropy_stats)

    if spec.kind == "static":
        tau = torch.full_like(h_norm, float(spec.static_tau), dtype=torch.float32)
        hard_mask = None

    elif spec.kind == "piecewise":
        tau = tau_piecewise(h_norm, spec)
        hard_mask = None

    elif spec.kind == "sigmoid":
        tau = tau_sigmoid(h_norm, spec)
        hard_mask = None

    elif spec.kind == "sigmoid_hard_gate":
        tau = tau_sigmoid(h_norm, spec)
        hard_mask = h_norm < float(spec.hard_gate)

    elif spec.kind == "soft_target_entropy":
        tau = full_vocab_target_entropy_for_batch(
            labels_shifted=labels_shifted,
            h_norm=h_norm,
            vocab_emb_norm=vocab_emb_norm,
            spec=spec,
            valid_mask=valid_mask,
            chunk_size=max(16, TRAIN_CFG.full_vocab_chunk_size),
        )
        hard_mask = None

    else:
        raise ValueError(f"Unsupported condition kind for tau: {spec.kind}")

    if spec.shuffled_tau:
        # For gated conditions, shuffle tau and hard/soft assignment together by encoding hard positions as 0.
        if hard_mask is not None:
            tau_or_zero = tau.clone()
            tau_or_zero[hard_mask] = 0.0
            shuffled = shuffle_flat_valid_values(tau_or_zero, valid_mask)
            hard_mask = shuffled <= 0.0
            tau = shuffled.clamp_min(1e-6)
        else:
            tau = shuffle_flat_valid_values(tau, valid_mask)

    return tau, hard_mask, valid_mask

def compute_condition_loss(
    model: GPT2LMHeadModel,
    batch: Dict[str, torch.Tensor],
    spec: ConditionSpec,
    reference_model: Optional[GPT2LMHeadModel] = None,
    entropy_stats: Optional[Dict[str, float]] = None,
    vocab_emb_norm: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch.get("attention_mask", None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits

    hard_sum, hard_count = hard_ce_loss_sum_count(logits, input_ids, attention_mask)
    hard_loss = hard_sum / hard_count.clamp_min(1.0)

    metrics = {
        "hard_ce_loss": float(hard_loss.detach().item()),
        "hard_tokens": float(hard_count.detach().item()),
    }

    if spec.kind == "hard":
        return hard_loss, metrics

    if reference_model is None or entropy_stats is None or vocab_emb_norm is None:
        raise ValueError(f"Condition {spec.name} requires reference_model, entropy_stats, and vocab_emb_norm.")

    tau, hard_mask, valid_mask = compute_tau_and_masks(
        spec=spec,
        reference_model=reference_model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        entropy_stats=entropy_stats,
        vocab_emb_norm=vocab_emb_norm,
    )

    metrics.update({
        "tau_mean": float(tau[valid_mask].mean().detach().item()) if valid_mask.any() else float("nan"),
        "tau_q50": float(tau[valid_mask].median().detach().item()) if valid_mask.any() else float("nan"),
    })

    if hard_mask is None:
        soft_sum, soft_count = full_vocab_soft_ce_loss_sum_count(
            student_logits=logits,
            input_ids=input_ids,
            attention_mask=attention_mask,
            vocab_emb_norm=vocab_emb_norm,
            tau=tau,
            token_mask=valid_mask,
            chunk_size=TRAIN_CFG.full_vocab_chunk_size,
        )
        soft_loss = soft_sum / soft_count.clamp_min(1.0)
        metrics["soft_loss"] = float(soft_loss.detach().item())
        metrics["soft_tokens"] = float(soft_count.detach().item())

        if spec.mix_alpha is None:
            return soft_loss, metrics

        loss = float(spec.mix_alpha) * hard_loss + (1.0 - float(spec.mix_alpha)) * soft_loss
        return loss, metrics

    # Gated setup.
    hard_positions = hard_mask & valid_mask
    soft_positions = (~hard_mask) & valid_mask

    gated_hard_sum, gated_hard_count = hard_ce_loss_sum_count(
        logits,
        input_ids,
        attention_mask,
        token_mask=hard_positions,
    )
    gated_soft_sum, gated_soft_count = full_vocab_soft_ce_loss_sum_count(
        student_logits=logits,
        input_ids=input_ids,
        attention_mask=attention_mask,
        vocab_emb_norm=vocab_emb_norm,
        tau=tau,
        token_mask=soft_positions,
        chunk_size=TRAIN_CFG.full_vocab_chunk_size,
    )

    gated_hard_loss = gated_hard_sum / gated_hard_count.clamp_min(1.0)
    gated_soft_loss = gated_soft_sum / gated_soft_count.clamp_min(1.0)

    metrics.update({
        "gated_hard_loss": float(gated_hard_loss.detach().item()) if gated_hard_count.item() > 0 else float("nan"),
        "gated_soft_loss": float(gated_soft_loss.detach().item()) if gated_soft_count.item() > 0 else float("nan"),
        "gated_hard_tokens": float(gated_hard_count.detach().item()),
        "gated_soft_tokens": float(gated_soft_count.detach().item()),
        "soft_fraction": float((gated_soft_count / (gated_hard_count + gated_soft_count).clamp_min(1.0)).detach().item()),
    })

    if spec.mix_alpha is None:
        total_sum = gated_hard_sum + gated_soft_sum
        total_count = gated_hard_count + gated_soft_count
        return total_sum / total_count.clamp_min(1.0), metrics

    # Mixed gated objective:
    # hard CE is computed over all positions; soft CE regularizes only high-entropy soft positions.
    if gated_soft_count.item() == 0:
        return hard_loss, metrics

    loss = float(spec.mix_alpha) * hard_loss + (1.0 - float(spec.mix_alpha)) * gated_soft_loss
    return loss, metrics


In [ ]:
# 16. Evaluation
# Simplified v

@torch.no_grad()
def evaluate_lm(
    model: GPT2LMHeadModel,
    dataloader,
    max_batches: Optional[int] = None,
) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_tokens = 0.0
    total_correct = 0.0

    for bidx, batch in enumerate(tqdm(dataloader, desc="Eval", leave=False)):
        if max_batches is not None and bidx >= max_batches:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        out = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = out.logits[:, :-1, :].contiguous()
        labels = input_ids[:, 1:].contiguous()
        mask = shift_attention_mask(attention_mask, labels)

        V = logits.size(-1)
        loss = F.cross_entropy(
            logits.reshape(-1, V).float(),
            labels.reshape(-1),
            reduction="none",
        ).view_as(labels)

        preds = logits.argmax(dim=-1)
        correct = (preds == labels).float()

        total_loss += float((loss * mask).sum().item())
        total_correct += float((correct * mask).sum().item())
        total_tokens += float(mask.sum().item())

    avg_loss = total_loss / max(total_tokens, 1.0)
    return {
        "val_loss": avg_loss,
        "val_ppl": math.exp(min(avg_loss, 20)),
        "val_accuracy": total_correct / max(total_tokens, 1.0),
        "val_tokens": int(total_tokens),
    }


In [ ]:
# 17. Save/load training checkpoints


def save_training_checkpoint(
    path: Path,
    model: GPT2LMHeadModel,
    optimizer=None,
    scheduler=None,
    scaler=None,
    global_step: int = 0,
    best_val_loss: float = float("inf"),
    bad_eval_count: int = 0,
    extra_state: Optional[Dict[str, Any]] = None,
):
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path, safe_serialization=True)
    tokenizer.save_pretrained(path)

    state = {
        "global_step": global_step,
        "best_val_loss": best_val_loss,
        "bad_eval_count": bad_eval_count,
        "rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
        "extra_state": extra_state or {},
    }
    if optimizer is not None:
        state["optimizer"] = optimizer.state_dict()
    if scheduler is not None:
        state["scheduler"] = scheduler.state_dict()
    if scaler is not None:
        state["scaler"] = scaler.state_dict()

    torch.save(state, path / "training_state.pt")

def load_training_state(path: Path, optimizer=None, scheduler=None, scaler=None):
    state_path = Path(path) / "training_state.pt"
    if not state_path.exists():
        return {
            "global_step": 0,
            "best_val_loss": float("inf"),
            "bad_eval_count": 0,
        }

    state = safe_torch_load(state_path, map_location="cpu")

    if optimizer is not None and "optimizer" in state:
        optimizer.load_state_dict(state["optimizer"])
    if scheduler is not None and "scheduler" in state:
        scheduler.load_state_dict(state["scheduler"])
    if scaler is not None and "scaler" in state:
        scaler.load_state_dict(state["scaler"])

    try:
        torch.set_rng_state(state["rng_state"])
        if torch.cuda.is_available() and state.get("cuda_rng_state_all") is not None:
            torch.cuda.set_rng_state_all(state["cuda_rng_state_all"])
        np.random.set_state(state["numpy_rng_state"])
        random.setstate(state["python_rng_state"])
    except Exception as e:
        print("Warning: failed to restore RNG state:", e)

    return state

def copy_checkpoint(src: Path, dst: Path):
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)


In [ ]:
# 18. Reference entropy stats and frozen reference


@torch.no_grad()
def compute_entropy_stats(reference_model, dataloader, max_batches=None) -> Dict[str, float]:
    reference_model.eval()
    values = []

    for bidx, batch in enumerate(tqdm(dataloader, desc="Computing reference entropy stats")):
        if max_batches is not None and bidx >= max_batches:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        ent = compute_reference_entropy(reference_model, input_ids, attention_mask)
        labels = input_ids[:, 1:].contiguous()
        mask = shift_attention_mask(attention_mask, labels).bool()
        values.append(ent[mask].detach().cpu())

    if not values:
        raise RuntimeError("No entropy values collected.")

    x = torch.cat(values).float().numpy()
    stats = {
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "q01": float(np.quantile(x, 0.01)),
        "q05": float(np.quantile(x, 0.05)),
        "q25": float(np.quantile(x, 0.25)),
        "q50": float(np.quantile(x, 0.50)),
        "q75": float(np.quantile(x, 0.75)),
        "q95": float(np.quantile(x, 0.95)),
        "q99": float(np.quantile(x, 0.99)),
        "n": int(len(x)),
    }
    return stats

def entropy_stats_cache_path(seed: int) -> Path:
    return CACHE_DIR / f"reference_entropy_stats_seed_{seed}.json"

def get_or_build_reference_signals(seed: int, signal_loader):
    '''
    Uses the selected hard_ce checkpoint as a frozen reference.
    This is deliberate: dynamic entropy should not come from the currently training student.
    '''
    hard_ckpt = selected_checkpoint_dir(seed, "hard_ce")
    if hard_ckpt is None:
        raise FileNotFoundError(f"No valid hard_ce checkpoint found for seed {seed}. Train hard_ce first.")

    reference_model = GPT2LMHeadModel.from_pretrained(hard_ckpt).to(device)
    reference_model.eval()
    for p in reference_model.parameters():
        p.requires_grad_(False)

    stats_path = entropy_stats_cache_path(seed)
    if stats_path.exists():
        entropy_stats = load_json(stats_path)
    else:
        entropy_stats = compute_entropy_stats(
            reference_model,
            signal_loader,
            max_batches=TRAIN_CFG.entropy_stats_max_batches,
        )
        save_json(entropy_stats, stats_path)

    vocab_emb_norm = get_vocab_emb_norm(reference_model).to(device)

    print("Reference checkpoint:", hard_ckpt)
    print("Entropy stats:", entropy_stats)
    return reference_model, entropy_stats, vocab_emb_norm


In [ ]:
# 19. Train one condition


def train_condition(
    seed: int,
    condition: str,
    train_loader,
    valid_loader,
    reference_model: Optional[GPT2LMHeadModel] = None,
    entropy_stats: Optional[Dict[str, float]] = None,
    vocab_emb_norm: Optional[torch.Tensor] = None,
):
    spec = CONDITION_SPECS[condition]
    run_dir = condition_root(seed, condition)
    run_dir.mkdir(parents=True, exist_ok=True)

    best_dir = best_checkpoint_dir(seed, condition)
    selected = selected_checkpoint_dir(seed, condition)

    if (
        SKIP_FINISHED
        and completion_marker_path(seed, condition).exists()
        and checkpoint_has_model_weights(best_dir)
    ):
        print(f"[skip] {condition} seed {seed}: already completed.")
        model = GPT2LMHeadModel.from_pretrained(best_dir).to(device)
        logs = pd.read_csv(log_path(seed, condition)) if log_path(seed, condition).exists() else pd.DataFrame()
        return model, logs, best_dir

    set_seed(seed)

    resume_dir = None
    if RESUME_IF_AVAILABLE:
        latest = latest_checkpoint_dir(seed, condition)
        if latest is not None and checkpoint_has_model_weights(latest):
            resume_dir = latest

    if resume_dir is not None:
        print(f"[resume] {condition} seed {seed} from {resume_dir}")
        model = GPT2LMHeadModel.from_pretrained(resume_dir).to(device)
    else:
        model = build_model_from_scratch().to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_CFG.learning_rate,
        weight_decay=TRAIN_CFG.weight_decay,
    )

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=TRAIN_CFG.warmup_steps,
        num_training_steps=TRAIN_CFG.max_steps,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda" and TRAIN_CFG.use_amp))

    start_step = 0
    best_val_loss = float("inf")
    bad_eval_count = 0
    logs = []

    if resume_dir is not None:
        state = load_training_state(resume_dir, optimizer=optimizer, scheduler=scheduler, scaler=scaler)
        start_step = int(state.get("global_step", 0))
        best_val_loss = float(state.get("best_val_loss", float("inf")))
        bad_eval_count = int(state.get("bad_eval_count", 0))

    if log_path(seed, condition).exists():
        try:
            old_logs = pd.read_csv(log_path(seed, condition)).to_dict("records")
            logs.extend(old_logs)
        except Exception:
            pass

    model.train()
    pbar = tqdm(total=TRAIN_CFG.max_steps, initial=start_step, desc=f"Training {condition} seed {seed}")

    global_step = start_step
    optimizer.zero_grad(set_to_none=True)

    while global_step < TRAIN_CFG.max_steps:
        for batch in train_loader:
            if global_step >= TRAIN_CFG.max_steps:
                break

            with maybe_autocast():
                loss, train_metrics = compute_condition_loss(
                    model=model,
                    batch=batch,
                    spec=spec,
                    reference_model=reference_model,
                    entropy_stats=entropy_stats,
                    vocab_emb_norm=vocab_emb_norm,
                )
                loss = loss / TRAIN_CFG.grad_accum_steps

            scaler.scale(loss).backward()

            if (global_step + 1) % TRAIN_CFG.grad_accum_steps == 0:
                if TRAIN_CFG.max_grad_norm is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CFG.max_grad_norm)

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            global_step += 1
            pbar.update(1)

            if global_step % TRAIN_CFG.eval_every == 0 or global_step == TRAIN_CFG.max_steps:
                metrics = evaluate_lm(model, valid_loader, max_batches=TRAIN_CFG.eval_max_batches)
                val_loss = metrics["val_loss"]

                row = {
                    "seed": seed,
                    "condition": condition,
                    "step": global_step,
                    "train_loss_last": float((loss * TRAIN_CFG.grad_accum_steps).detach().item()),
                    "lr": float(scheduler.get_last_lr()[0]),
                    **metrics,
                    **train_metrics,
                }
                logs.append(row)
                pd.DataFrame(logs).to_csv(log_path(seed, condition), index=False)

                print(json.dumps(row, indent=2))

                if val_loss < best_val_loss - TRAIN_CFG.early_stopping_min_delta:
                    best_val_loss = val_loss
                    bad_eval_count = 0
                    tmp_dir = step_checkpoint_dir(seed, condition, global_step)
                    save_training_checkpoint(
                        tmp_dir,
                        model,
                        optimizer=optimizer,
                        scheduler=scheduler,
                        scaler=scaler,
                        global_step=global_step,
                        best_val_loss=best_val_loss,
                        bad_eval_count=bad_eval_count,
                    )
                    copy_checkpoint(tmp_dir, best_dir)
                    print(f"[best] {condition} seed {seed}: val_loss={best_val_loss:.4f}")
                else:
                    bad_eval_count += 1

                if (
                    TRAIN_CFG.early_stopping_patience is not None
                    and global_step >= TRAIN_CFG.early_stopping_min_steps
                    and bad_eval_count >= TRAIN_CFG.early_stopping_patience
                ):
                    print(f"[early stop] {condition} seed {seed} at step {global_step}")
                    global_step = TRAIN_CFG.max_steps
                    break

            if global_step % TRAIN_CFG.save_every == 0:
                ckpt_dir = step_checkpoint_dir(seed, condition, global_step)
                save_training_checkpoint(
                    ckpt_dir,
                    model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    scaler=scaler,
                    global_step=global_step,
                    best_val_loss=best_val_loss,
                    bad_eval_count=bad_eval_count,
                )

        # If train_loader is exhausted before max_steps, loop over it again.
        if global_step >= TRAIN_CFG.max_steps:
            break

    pbar.close()

    final_dir = final_checkpoint_dir(seed, condition)
    save_training_checkpoint(
        final_dir,
        model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        global_step=global_step,
        best_val_loss=best_val_loss,
        bad_eval_count=bad_eval_count,
    )

    if not checkpoint_has_model_weights(best_dir):
        print("[warn] best checkpoint missing; copying final to best.")
        copy_checkpoint(final_dir, best_dir)

    selected_dir = selected_checkpoint_dir(seed, condition)

    # Fast final eval on the current final model, consistent with training-time eval.
    final_eval_fast = evaluate_lm(model, valid_loader, max_batches=TRAIN_CFG.eval_max_batches)

    # Full validation once on the selected checkpoint.
    # This is the number to use for final tables.
    selected_full_eval = {}
    if selected_dir is not None and checkpoint_has_model_weights(selected_dir):
        print(f"[full final eval] selected checkpoint for {condition} seed {seed}: {selected_dir}")
        selected_model = GPT2LMHeadModel.from_pretrained(selected_dir).to(device)
        selected_full_eval = evaluate_lm(selected_model, valid_loader, max_batches=None)
        del selected_model
        cleanup_cuda()
    else:
        print("[warn] No selected checkpoint found for full final eval.")

    summary = {
        "seed": seed,
        "condition": condition,
        "short_name": CONDITION_ALIASES.get(condition, condition),
        "spec": asdict(spec),
        "global_step": global_step,
        "best_val_loss_train_eval": best_val_loss,
        "selected_checkpoint": str(selected_dir),
        "final_eval_fast": final_eval_fast,
        "selected_full_eval": selected_full_eval,
        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    save_json(summary, summary_path(seed, condition))
    save_json(summary, completion_marker_path(seed, condition))

    return model, pd.DataFrame(logs), selected_dir


In [ ]:
# 20. Run full experiment


if RUN_FULL_EXPERIMENT:
    all_run_summaries = []

    for seed in SEEDS:
        print("=" * 100)
        print(f"Starting seed {seed}")
        print("=" * 100)

        # 1. Train hard CE first, with fresh shuffled dataloader.
        train_loader, valid_loader = make_dataloaders(seed, train_shuffle=True)
        hard_model, hard_log, hard_selected_path = train_condition(
            seed=seed,
            condition="hard_ce",
            train_loader=train_loader,
            valid_loader=valid_loader,
        )
        all_run_summaries.append({
            "seed": seed,
            "condition": "hard_ce",
            "selected_checkpoint": str(hard_selected_path),
        })
        del hard_model
        cleanup_cuda()

        # 2. Build frozen reference signals from hard CE.
        # Use non-shuffled loader for entropy stats.
        signal_loader, _ = make_dataloaders(seed, train_shuffle=False)
        reference_model, entropy_stats, vocab_emb_norm = get_or_build_reference_signals(seed, signal_loader)

        # 3. Train all non-hard conditions. Each gets a fresh dataloader with the same seed.
        for condition in CONDITIONS:
            if condition == "hard_ce":
                continue

            print("=" * 100)
            print(f"Training condition: {condition}, seed {seed}")
            print("=" * 100)

            train_loader, valid_loader = make_dataloaders(seed, train_shuffle=True)

            model, log_df, selected_path = train_condition(
                seed=seed,
                condition=condition,
                train_loader=train_loader,
                valid_loader=valid_loader,
                reference_model=reference_model,
                entropy_stats=entropy_stats,
                vocab_emb_norm=vocab_emb_norm,
            )

            all_run_summaries.append({
                "seed": seed,
                "condition": condition,
                "selected_checkpoint": str(selected_path),
            })

            del model
            cleanup_cuda()

        del reference_model, vocab_emb_norm
        cleanup_cuda()

    summary_df = pd.DataFrame(all_run_summaries)
    out_path = LOG_DIR / "all_run_summaries.csv"
    summary_df.to_csv(out_path, index=False)
    print("Saved:", out_path)
    display(summary_df)
else:
    print("RUN_FULL_EXPERIMENT is False. Set it to True when ready.")
    print("Current stage:", EXPERIMENT_STAGE)
    print("Conditions:", CONDITIONS)


Starting seed 42
[skip] hard_ce seed 42: already completed.
Reference checkpoint: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/checkpoints/seed_42/hard_ce/best_by_val_loss
Entropy stats: {'mean': 3.8352062702178955, 'std': 2.2439565658569336, 'q01': 0.0027693677693605423, 'q05': 0.006557294633239508, 'q25': 2.4359638690948486, 'q50': 4.047797679901123, 'q75': 5.367341041564941, 'q95': 7.346242427825928, 'q99': 8.066534042358398, 'n': 16135635}
Training condition: full_vocab_static_tau_0p055, seed 42
[skip] full_vocab_static_tau_0p055 seed 42: already completed.
Training condition: full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9, seed 42
[skip] full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9 seed 42: already completed.
Training condition: full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9_shuffled_tau, seed 42
[skip] full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0

,seed,condition,selected_checkpoint
0,42,hard_ce,/content/drive/MyDrive/babylm_full_vocab_soft_...
1,42,full_vocab_static_tau_0p055,/content/drive/MyDrive/babylm_full_vocab_soft_...
2,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,/content/drive/MyDrive/babylm_full_vocab_soft_...
3,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,/content/drive/MyDrive/babylm_full_vocab_soft_...
4,42,full_vocab_piecewise_tau_max_0p055_mix_alpha_0p9,/content/drive/MyDrive/babylm_full_vocab_soft_...
5,43,hard_ce,/content/drive/MyDrive/babylm_full_vocab_soft_...
6,43,full_vocab_static_tau_0p055,/content/drive/MyDrive/babylm_full_vocab_soft_...
7,43,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,/content/drive/MyDrive/babylm_full_vocab_soft_...
8,43,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,/content/drive/MyDrive/babylm_full_vocab_soft_...
9,43,full_vocab_piecewise_tau_max_0p055_mix_alpha_0p9,/content/drive/MyDrive/babylm_full_vocab_soft_...


In [ ]:
# 21. Aggregate training summaries


def build_checkpoint_index():
    rows = []
    for seed in SEEDS:
        for condition in CONDITIONS:
            root = condition_root(seed, condition)
            short = CONDITION_ALIASES.get(condition, condition)

            for ckpt in list_condition_checkpoints(seed, condition):
                rows.append({
                    "seed": seed,
                    "condition": condition,
                    "short_name": short,
                    "checkpoint_type": "step",
                    "step": parse_step_from_path(ckpt),
                    "checkpoint": str(ckpt),
                })

            for ctype, ckpt in [
                ("best_by_val_loss", best_checkpoint_dir(seed, condition)),
                ("final", final_checkpoint_dir(seed, condition)),
            ]:
                if checkpoint_has_model_weights(ckpt):
                    rows.append({
                        "seed": seed,
                        "condition": condition,
                        "short_name": short,
                        "checkpoint_type": ctype,
                        "step": None,
                        "checkpoint": str(ckpt),
                    })

            selected = selected_checkpoint_dir(seed, condition)
            if selected is not None and checkpoint_has_model_weights(selected):
                rows.append({
                    "seed": seed,
                    "condition": condition,
                    "short_name": short,
                    "checkpoint_type": "selected",
                    "step": None,
                    "checkpoint": str(selected),
                })

    df = pd.DataFrame(rows)
    out = LOG_DIR / "checkpoint_index.csv"
    df.to_csv(out, index=False)
    print("Saved:", out)
    return df

def aggregate_training_summaries():
    rows = []
    for seed in SEEDS:
        for condition in CONDITIONS:
            sp = summary_path(seed, condition)
            if not sp.exists():
                continue
            s = load_json(sp)
            full_eval = s.get("selected_full_eval", {})
            fast_eval = s.get("final_eval_fast", {})
            rows.append({
                "seed": seed,
                "condition": condition,
                "short_name": s.get("short_name", CONDITION_ALIASES.get(condition, condition)),
                "best_val_loss_train_eval": s.get("best_val_loss_train_eval", np.nan),
                "selected_checkpoint": s.get("selected_checkpoint", None),
                **{f"selected_full_{k}": v for k, v in full_eval.items()},
                **{f"final_fast_{k}": v for k, v in fast_eval.items()},
            })

    df = pd.DataFrame(rows)
    if len(df):
        out = LOG_DIR / "aggregate_training_summary.csv"
        df.to_csv(out, index=False)
        print("Saved:", out)
        sort_col = "selected_full_val_loss" if "selected_full_val_loss" in df.columns else "best_val_loss_train_eval"
        display(df.sort_values(["seed", sort_col]))
    else:
        print("No summaries found.")

    ckpt_index = build_checkpoint_index()
    return df, ckpt_index

aggregate_df, checkpoint_index_df = aggregate_training_summaries()


Saved: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/logs/aggregate_training_summary.csv


,seed,condition,short_name,best_val_loss_train_eval,selected_checkpoint,selected_full_val_loss,selected_full_val_ppl,selected_full_val_accuracy,selected_full_val_tokens,final_fast_val_loss,final_fast_val_ppl,final_fast_val_accuracy,final_fast_val_tokens
0,42,hard_ce,hard_ce,3.767540,/content/drive/MyDrive/babylm_full_vocab_soft_...,3.724966,41.469819,0.376276,164220,3.767540,43.273474,0.371235,51000
2,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,sigmoid_hard_gate,3.766681,/content/drive/MyDrive/babylm_full_vocab_soft_...,3.725053,41.473429,0.375734,164220,3.766681,43.236334,0.370039,51000
3,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,sigmoid_hard_gate_shuffled,3.769402,/content/drive/MyDrive/babylm_full_vocab_soft_...,3.726519,41.534287,0.375606,164220,3.769402,43.354111,0.370333,51000
4,42,full_vocab_piecewise_tau_max_0p055_mix_alpha_0p9,piecewise_entropy,3.767752,/content/drive/MyDrive/babylm_full_vocab_soft_...,3.727122,41.559344,0.375685,164220,3.767752,43.282658,0.370373,51000
1,42,full_vocab_static_tau_0p055,static_soft,3.843406,/content/drive/MyDrive/babylm_full_vocab_soft_...,3.798548,44.636318,0.374906,164220,3.843406,46.684211,0.370333,51000
5,43,hard_ce,hard_ce,3.769639,/content/drive/MyDrive/babylm_full_vocab_soft_...,3.725920,41.509406,0.376531,164220,3.769639,43.364423,0.370686,51000
8,43,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,sigmoid_hard_gate_shuffled,3.771910,/content/drive/MyDrive/babylm_full_vocab_soft_...,3.728131,41.601286,0.375916,164220,3.771910,43.463020,0.369902,51000
7,43,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,sigmoid_hard_gate,3.772313,/content/drive/MyDrive/babylm_full_vocab_soft_...,3.729750,41.668680,0.375728,164220,3.772313,43.480528,0.369627,51000
9,43,full_vocab_piecewise_tau_max_0p055_mix_alpha_0p9,piecewise_entropy,3.779393,/content/drive/MyDrive/babylm_full_vocab_soft_...,3.734823,41.880614,0.376294,164220,3.779393,43.789446,0.369922,51000
6,43,full_vocab_static_tau_0p055,static_soft,3.845707,/content/drive/MyDrive/babylm_full_vocab_soft_...,3.798807,44.647904,0.374985,164220,3.845707,46.791734,0.368902,51000


Saved: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/logs/checkpoint_index.csv


In [ ]:
# 22. Tau/target diagnostics

@torch.no_grad()
def inspect_full_vocab_targets_for_batch(
    reference_model,
    batch,
    spec: ConditionSpec,
    entropy_stats,
    vocab_emb_norm,
    top_n: int = 10,
    max_examples: int = 20,
):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch.get("attention_mask", None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    labels = input_ids[:, 1:].contiguous()
    valid_mask = shift_attention_mask(attention_mask, labels).bool()
    tau, hard_mask, valid_mask = compute_tau_and_masks(
        spec, reference_model, input_ids, attention_mask, entropy_stats, vocab_emb_norm
    )
    ref_entropy = compute_reference_entropy(reference_model, input_ids, attention_mask)
    h_norm = normalize_entropy(ref_entropy, entropy_stats)

    flat_idx = torch.nonzero(valid_mask.reshape(-1), as_tuple=False).squeeze(-1)
    if flat_idx.numel() > max_examples:
        flat_idx = flat_idx[torch.randperm(flat_idx.numel(), device=flat_idx.device)[:max_examples]]

    E = vocab_emb_norm.to(device).float()
    labels_flat = labels.reshape(-1)
    tau_flat = tau.reshape(-1)
    h_flat = h_norm.reshape(-1)
    ent_flat = ref_entropy.reshape(-1)

    rows = []
    for idx in flat_idx:
        y = int(labels_flat[idx].item())
        tau_y = float(tau_flat[idx].item())
        emb = E[y:y+1]
        sims = (emb @ E.T).squeeze(0)
        probs = F.softmax(sims / max(tau_y, 1e-6), dim=-1)
        top_probs, top_ids = torch.topk(probs, k=top_n)

        rows.append({
            "target_id": y,
            "target_token": tokenizer.decode([y]),
            "tau": tau_y,
            "reference_entropy": float(ent_flat[idx].item()),
            "h_norm": float(h_flat[idx].item()),
            "gold_prob_q_y": float(probs[y].item()),
            "target_entropy": float((-(probs * probs.clamp_min(1e-12).log()).sum()).item()),
            "top_tokens": " | ".join([repr(tokenizer.decode([int(t)])) for t in top_ids.detach().cpu()]),
            "top_probs": " | ".join([f"{float(p):.4f}" for p in top_probs.detach().cpu()]),
        })
    return pd.DataFrame(rows)

@torch.no_grad()
def tau_distribution_diagnostic(
    seed: int,
    condition: str,
    max_batches: Optional[int] = None,
):
    spec = CONDITION_SPECS[condition]
    if spec.kind == "hard":
        return pd.DataFrame()

    signal_loader, _ = make_dataloaders(seed, train_shuffle=False)
    reference_model, entropy_stats, vocab_emb_norm = get_or_build_reference_signals(seed, signal_loader)

    values = []
    qy_values = []
    target_ent_values = []
    soft_fracs = []

    E = vocab_emb_norm.to(device).float()

    for bidx, batch in enumerate(tqdm(signal_loader, desc=f"Tau diagnostic {condition}")):
        if max_batches is not None and bidx >= max_batches:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        labels = input_ids[:, 1:].contiguous()
        tau, hard_mask, valid_mask = compute_tau_and_masks(
            spec, reference_model, input_ids, attention_mask, entropy_stats, vocab_emb_norm
        )

        tau_valid = tau[valid_mask].detach()
        values.append(tau_valid.cpu())

        if hard_mask is not None:
            soft_fracs.append(float(((~hard_mask) & valid_mask).float().sum().item() / max(valid_mask.float().sum().item(), 1.0)))

        # Estimate q(y) and target entropy on a small subset to keep diagnostic cheap.
        flat_valid = torch.nonzero(valid_mask.reshape(-1), as_tuple=False).squeeze(-1)
        if flat_valid.numel() > 64:
            flat_valid = flat_valid[torch.randperm(flat_valid.numel(), device=flat_valid.device)[:64]]

        labels_flat = labels.reshape(-1)
        tau_flat = tau.reshape(-1)
        for idx in flat_valid:
            y = labels_flat[idx]
            tau_y = tau_flat[idx].clamp_min(1e-6)
            sims = E[y:y+1] @ E.T
            probs = F.softmax(sims.squeeze(0) / tau_y, dim=-1)
            qy_values.append(float(probs[y].item()))
            target_ent_values.append(float((-(probs * probs.clamp_min(1e-12).log()).sum()).item()))

    tau_all = torch.cat(values).numpy() if values else np.array([])
    row = {
        "seed": seed,
        "condition": condition,
        "n_tau": int(len(tau_all)),
        "tau_mean": float(np.mean(tau_all)) if len(tau_all) else np.nan,
        "tau_q05": float(np.quantile(tau_all, 0.05)) if len(tau_all) else np.nan,
        "tau_q25": float(np.quantile(tau_all, 0.25)) if len(tau_all) else np.nan,
        "tau_q50": float(np.quantile(tau_all, 0.50)) if len(tau_all) else np.nan,
        "tau_q75": float(np.quantile(tau_all, 0.75)) if len(tau_all) else np.nan,
        "tau_q95": float(np.quantile(tau_all, 0.95)) if len(tau_all) else np.nan,
        "gold_prob_q_y_mean": float(np.mean(qy_values)) if qy_values else np.nan,
        "target_entropy_mean": float(np.mean(target_ent_values)) if target_ent_values else np.nan,
        "soft_fraction_mean": float(np.mean(soft_fracs)) if soft_fracs else np.nan,
    }

    del reference_model, vocab_emb_norm
    cleanup_cuda()
    return pd.DataFrame([row])

if RUN_TAU_DIAGNOSTICS:
    rows = []
    for seed in SEEDS:
        for condition in CONDITIONS:
            if CONDITION_SPECS[condition].kind == "hard":
                continue
            rows.append(tau_distribution_diagnostic(seed, condition, max_batches=TRAIN_CFG.entropy_stats_max_batches))
    tau_diag = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    out = DIAG_DIR / "tau_distribution_diagnostic.csv"
    tau_diag.to_csv(out, index=False)
    print("Saved:", out)
    display(tau_diag)
else:
    print("Tau diagnostics disabled. Set RUN_TAU_DIAGNOSTICS = True to run.")


Reference checkpoint: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/checkpoints/seed_42/hard_ce/best_by_val_loss
Entropy stats: {'mean': 3.8352062702178955, 'std': 2.2439565658569336, 'q01': 0.0027693677693605423, 'q05': 0.006557294633239508, 'q25': 2.4359638690948486, 'q50': 4.047797679901123, 'q75': 5.367341041564941, 'q95': 7.346242427825928, 'q99': 8.066534042358398, 'n': 16135635}


Tau diagnostic full_vocab_static_tau_0p055:   0%|          | 0/7910 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# 23. Entropy bucket evaluation


@torch.no_grad()
def entropy_bucket_eval_for_model(
    model,
    reference_model,
    entropy_stats,
    dataloader,
    max_batches=None,
):
    model.eval()
    buckets = {
        "low": {"loss_sum": 0.0, "correct": 0.0, "tokens": 0.0},
        "mid": {"loss_sum": 0.0, "correct": 0.0, "tokens": 0.0},
        "high": {"loss_sum": 0.0, "correct": 0.0, "tokens": 0.0},
    }

    for bidx, batch in enumerate(tqdm(dataloader, desc="Entropy bucket eval", leave=False)):
        if max_batches is not None and bidx >= max_batches:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1, :].float()
        labels = input_ids[:, 1:].contiguous()
        mask = shift_attention_mask(attention_mask, labels).bool()

        V = logits.size(-1)
        loss = F.cross_entropy(logits.reshape(-1, V), labels.reshape(-1), reduction="none").view_as(labels)
        pred = logits.argmax(dim=-1)
        correct = pred == labels

        ref_entropy = compute_reference_entropy(reference_model, input_ids, attention_mask)
        h_norm = normalize_entropy(ref_entropy, entropy_stats)

        bucket_masks = {
            "low": (h_norm < 0.33) & mask,
            "mid": (h_norm >= 0.33) & (h_norm < 0.66) & mask,
            "high": (h_norm >= 0.66) & mask,
        }

        for name, bm in bucket_masks.items():
            if bm.sum().item() == 0:
                continue
            buckets[name]["loss_sum"] += float(loss[bm].sum().item())
            buckets[name]["correct"] += float(correct[bm].float().sum().item())
            buckets[name]["tokens"] += float(bm.sum().item())

    rows = []
    for name, b in buckets.items():
        tokens = max(b["tokens"], 1.0)
        avg_loss = b["loss_sum"] / tokens
        rows.append({
            "bucket": name,
            "tokens": int(b["tokens"]),
            "loss": avg_loss,
            "ppl": math.exp(min(avg_loss, 20)),
            "accuracy": b["correct"] / tokens,
        })
    return pd.DataFrame(rows)

if RUN_ENTROPY_BUCKET_EVAL:
    rows = []
    for seed in SEEDS:
        _, valid_loader = make_dataloaders(seed, train_shuffle=False)
        signal_loader, _ = make_dataloaders(seed, train_shuffle=False)
        reference_model, entropy_stats, vocab_emb_norm = get_or_build_reference_signals(seed, signal_loader)

        for condition in CONDITIONS:
            ckpt = selected_checkpoint_dir(seed, condition)
            if ckpt is None:
                print("Missing checkpoint:", seed, condition)
                continue
            model = GPT2LMHeadModel.from_pretrained(ckpt).to(device)
            df = entropy_bucket_eval_for_model(
                model,
                reference_model,
                entropy_stats,
                valid_loader,
                max_batches=TRAIN_CFG.eval_max_batches,
            )
            df.insert(0, "condition", condition)
            df.insert(0, "seed", seed)
            rows.append(df)
            del model
            cleanup_cuda()

        del reference_model, vocab_emb_norm
        cleanup_cuda()

    bucket_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    out = DIAG_DIR / "entropy_bucket_eval.csv"
    bucket_df.to_csv(out, index=False)
    print("Saved:", out)
    display(bucket_df)
else:
    print("Entropy bucket eval disabled. Set RUN_ENTROPY_BUCKET_EVAL = True to run.")


Reference checkpoint: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/checkpoints/seed_42/hard_ce/best_by_val_loss
Entropy stats: {'mean': 3.8352062702178955, 'std': 2.2439565658569336, 'q01': 0.0027693677693605423, 'q05': 0.006557294633239508, 'q25': 2.4359638690948486, 'q50': 4.047797679901123, 'q75': 5.367341041564941, 'q95': 7.346242427825928, 'q99': 8.066534042358398, 'n': 16135635}


Entropy bucket eval:   0%|          | 0/81 [00:00<?, ?it/s]

Entropy bucket eval:   0%|          | 0/81 [00:00<?, ?it/s]

Entropy bucket eval:   0%|          | 0/81 [00:00<?, ?it/s]

Entropy bucket eval:   0%|          | 0/81 [00:00<?, ?it/s]

...

Entropy bucket eval:   0%|          | 0/81 [00:00<?, ?it/s]

Entropy bucket eval:   0%|          | 0/81 [00:00<?, ?it/s]

Entropy bucket eval:   0%|          | 0/81 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/diagnostics/entropy_bucket_eval.csv


,seed,condition,bucket,tokens,loss,ppl,accuracy
0,42,hard_ce,low,12213,0.602465,1.826617,0.851797
1,42,hard_ce,mid,20149,3.693767,40.195962,0.313117
2,42,hard_ce,high,18638,5.921286,372.890816,0.119165
3,42,full_vocab_static_tau_0p055,low,12213,0.624503,1.867318,0.852862
4,42,full_vocab_static_tau_0p055,mid,20149,3.723775,41.420443,0.312025
5,42,full_vocab_static_tau_0p055,high,18638,6.082000,437.904091,0.117180
6,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,low,12213,0.606958,1.834841,0.849832
7,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,mid,20149,3.695772,40.276668,0.311628
8,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,high,18638,5.913824,370.118653,0.118790
9,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,low,12213,0.604366,1.830091,0.850978


In [ ]:
# 24. Embedding-distance / IV-style diagnostic


@torch.no_grad()
def embedding_distance_diagnostic(
    model,
    reference_model,
    dataloader,
    pred_top_k: int = 20,
    max_batches: Optional[int] = None,
):
    model.eval()
    E = get_vocab_emb_norm(reference_model).to(device).float()

    total_expected_distance = 0.0
    total_nll = 0.0
    total_tokens = 0.0

    for bidx, batch in enumerate(tqdm(dataloader, desc="Embedding distance diagnostic", leave=False)):
        if max_batches is not None and bidx >= max_batches:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1, :].float()
        labels = input_ids[:, 1:].contiguous()
        mask = shift_attention_mask(attention_mask, labels).bool()

        log_probs = F.log_softmax(logits, dim=-1)
        probs_top, ids_top = torch.topk(torch.exp(log_probs), k=pred_top_k, dim=-1)
        probs_top = probs_top / probs_top.sum(dim=-1, keepdim=True).clamp_min(1e-12)

        true_emb = E[labels]
        pred_emb = E[ids_top]
        cos = (pred_emb * true_emb.unsqueeze(-2)).sum(dim=-1)
        distances = 1.0 - cos

        expected_distance = (probs_top * distances).sum(dim=-1)
        nll = -log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

        total_expected_distance += float(expected_distance[mask].sum().item())
        total_nll += float(nll[mask].sum().item())
        total_tokens += float(mask.sum().item())

    return {
        "embedding_expected_distance_topk": total_expected_distance / max(total_tokens, 1.0),
        "avg_nll": total_nll / max(total_tokens, 1.0),
        "tokens": int(total_tokens),
        "pred_top_k": pred_top_k,
    }

if RUN_EMBEDDING_DISTANCE_DIAG:
    rows = []
    for seed in SEEDS:
        _, valid_loader = make_dataloaders(seed, train_shuffle=False)
        signal_loader, _ = make_dataloaders(seed, train_shuffle=False)
        reference_model, entropy_stats, vocab_emb_norm = get_or_build_reference_signals(seed, signal_loader)

        for condition in CONDITIONS:
            ckpt = selected_checkpoint_dir(seed, condition)
            if ckpt is None:
                continue
            model = GPT2LMHeadModel.from_pretrained(ckpt).to(device)
            metrics = embedding_distance_diagnostic(
                model,
                reference_model,
                valid_loader,
                pred_top_k=20,
                max_batches=TRAIN_CFG.eval_max_batches,
            )
            rows.append({"seed": seed, "condition": condition, **metrics})
            del model
            cleanup_cuda()

        del reference_model, vocab_emb_norm
        cleanup_cuda()

    iv_df = pd.DataFrame(rows)
    out = DIAG_DIR / "embedding_distance_iv_diagnostic.csv"
    iv_df.to_csv(out, index=False)
    print("Saved:", out)
    display(iv_df)
else:
    print("Embedding-distance diagnostic disabled. Set RUN_EMBEDDING_DISTANCE_DIAG = True to run.")


Reference checkpoint: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/checkpoints/seed_42/hard_ce/best_by_val_loss
Entropy stats: {'mean': 3.8352062702178955, 'std': 2.2439565658569336, 'q01': 0.0027693677693605423, 'q05': 0.006557294633239508, 'q25': 2.4359638690948486, 'q50': 4.047797679901123, 'q75': 5.367341041564941, 'q95': 7.346242427825928, 'q99': 8.066534042358398, 'n': 16135635}


Embedding distance diagnostic:   0%|          | 0/81 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7da89cd70b80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 415, in select
    fd_event_list = self._selector.poll(timeout)
    

Embedding distance diagnostic:   0%|          | 0/81 [00:00<?, ?it/s]

Embedding distance diagnostic:   0%|          | 0/81 [00:00<?, ?it/s]

...

Embedding distance diagnostic:   0%|          | 0/81 [00:00<?, ?it/s]

Embedding distance diagnostic:   0%|          | 0/81 [00:00<?, ?it/s]

Embedding distance diagnostic:   0%|          | 0/81 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/diagnostics/embedding_distance_iv_diagnostic.csv


,seed,condition,embedding_expected_distance_topk,avg_nll,tokens,pred_top_k
0,42,hard_ce,0.456426,3.767540,51000,20
1,42,full_vocab_static_tau_0p055,0.460295,3.843406,51000,20
2,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,0.456308,3.766681,51000,20
3,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,0.455914,3.769402,51000,20
4,42,full_vocab_piecewise_tau_max_0p055_mix_alpha_0p9,0.456525,3.767752,51000,20
5,43,hard_ce,0.456717,3.769639,51000,20
6,43,full_vocab_static_tau_0p055,0.461214,3.845707,51000,20
7,43,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,0.457446,3.772313,51000,20
8,43,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,0.456654,3.771910,51000,20
9,43,full_vocab_piecewise_tau_max_0p055_mix_alpha_0p9,0.455383,3.779393,51000,20


In [ ]:
# 25. Cloze evaluator with leading-space fix


def normalize_candidate_spacing(context: str, candidate: str) -> str:
    context = "" if pd.isna(context) else str(context)
    candidate = "" if pd.isna(candidate) else str(candidate)

    no_space_prefix = (" ", "\n", "\t", ".", ",", "!", "?", ":", ";", "'", '"', ")", "]", "}", "%")
    if (
        len(context) > 0
        and len(candidate) > 0
        and not context.endswith((" ", "\n", "\t"))
        and not candidate.startswith(no_space_prefix)
    ):
        return " " + candidate
    return candidate

@torch.no_grad()
def score_candidate_avg_logprob(model, context: str, candidate: str):
    model.eval()

    context = "" if pd.isna(context) else str(context)
    candidate_text = normalize_candidate_spacing(context, candidate)

    context_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    cand_ids = tokenizer(candidate_text, add_special_tokens=False)["input_ids"]

    if len(cand_ids) == 0:
        return -1e9

    # Keep candidate intact when truncating context.
    max_len = MODEL_CFG.block_size
    if len(cand_ids) >= max_len:
        return -1e9

    available_context_len = max_len - len(cand_ids)
    context_ids = context_ids[-available_context_len:]
    full_ids = context_ids + cand_ids

    input_ids = torch.tensor([full_ids], dtype=torch.long, device=device)
    attention_mask = torch.ones_like(input_ids)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    log_probs = F.log_softmax(logits.float(), dim=-1)

    cand_start = len(full_ids) - len(cand_ids)
    scores = []
    for j, tok in enumerate(cand_ids):
        pos = cand_start + j - 1
        if 0 <= pos < log_probs.size(1):
            scores.append(float(log_probs[0, pos, tok].item()))

    if not scores:
        return -1e9
    return float(np.mean(scores))

def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x)
    e = np.exp(x)
    return e / max(e.sum(), 1e-12)

def safe_pearson(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def run_cloze_long_format(model, cloze_df: pd.DataFrame):
    '''
    Preferred long format columns:
        item_id, context, candidate, human_prob
    Optional:
        target
    '''
    required = {"context", "candidate", "human_prob"}
    if not required.issubset(set(cloze_df.columns)):
        raise ValueError(f"Long cloze format needs columns: {required}")

    if "item_id" not in cloze_df.columns:
        cloze_df = cloze_df.copy()
        cloze_df["item_id"] = cloze_df.groupby("context").ngroup()

    rows = []
    for item_id, group in tqdm(cloze_df.groupby("item_id"), desc="Cloze items"):
        context = str(group["context"].iloc[0])
        candidates = [str(x) for x in group["candidate"].tolist()]
        human = np.asarray(group["human_prob"].astype(float).tolist(), dtype=np.float64)
        human = human / max(human.sum(), 1e-12)

        scores = np.asarray([score_candidate_avg_logprob(model, context, cand) for cand in candidates], dtype=np.float64)
        model_prob = softmax_np(scores)

        human_top = candidates[int(np.argmax(human))]
        model_top = candidates[int(np.argmax(model_prob))]
        item_pearson = safe_pearson(human, model_prob)
        kl = float(np.sum(human * (np.log(human.clip(1e-12)) - np.log(model_prob.clip(1e-12)))))
        human_entropy = float(-np.sum(human * np.log(human.clip(1e-12))))

        for cand, h, s, mp in zip(candidates, human, scores, model_prob):
            rows.append({
                "item_id": item_id,
                "context": context,
                "candidate": cand,
                "human_prob": float(h),
                "model_score": float(s),
                "model_prob": float(mp),
                "human_top": human_top,
                "model_top": model_top,
                "top1_correct": model_top == human_top,
                "item_pearson": item_pearson,
                "kl_human_model": kl,
                "human_entropy": human_entropy,
            })
    return pd.DataFrame(rows)

def run_cloze_wide_format(model, cloze_df: pd.DataFrame):
    '''
    Basic wide format:
        context, target, distractor1, distractor2, ...
    No human distribution is available, so only top-1 accuracy is reported.
    '''
    distractor_cols = [c for c in cloze_df.columns if c.startswith("distractor")]
    rows = []

    for idx, row in tqdm(cloze_df.iterrows(), total=len(cloze_df), desc="Cloze wide"):
        context = str(row["context"])
        target = str(row["target"])
        candidates = [target] + [str(row[c]) for c in distractor_cols if pd.notna(row[c])]
        scores = {cand: score_candidate_avg_logprob(model, context, cand) for cand in candidates}
        pred = max(scores, key=scores.get)

        for cand, score in scores.items():
            rows.append({
                "item_id": idx,
                "context": context,
                "candidate": cand,
                "target": target,
                "model_score": score,
                "prediction": pred,
                "correct": pred == target,
            })

    return pd.DataFrame(rows)

def summarize_cloze(result_df: pd.DataFrame):
    if "human_prob" in result_df.columns:
        candidate_pearson = safe_pearson(result_df["human_prob"], result_df["model_prob"])
        item_summary = result_df.groupby("item_id").agg({
            "item_pearson": "first",
            "kl_human_model": "first",
            "top1_correct": "first",
            "human_entropy": "first",
        })
        return {
            "candidate_pearson": candidate_pearson,
            "mean_item_pearson": float(item_summary["item_pearson"].mean()),
            "mean_kl": float(item_summary["kl_human_model"].mean()),
            "top1_accuracy": float(item_summary["top1_correct"].mean()),
            "n_items": int(len(item_summary)),
        }
    else:
        item_summary = result_df.groupby("item_id").agg({"correct": "first"})
        return {
            "top1_accuracy": float(item_summary["correct"].mean()),
            "n_items": int(len(item_summary)),
        }

if RUN_CLOZE_EVAL:
    cloze_df = pd.read_csv(CLOZE_CSV_PATH)
    all_rows = []
    summary_rows = []

    long_format = {"context", "candidate", "human_prob"}.issubset(set(cloze_df.columns))

    for seed in SEEDS:
        for condition in CONDITIONS:
            ckpt = selected_checkpoint_dir(seed, condition)
            if ckpt is None:
                continue
            print("Cloze eval:", seed, condition)
            model = GPT2LMHeadModel.from_pretrained(ckpt).to(device)

            if long_format:
                result = run_cloze_long_format(model, cloze_df)
            else:
                result = run_cloze_wide_format(model, cloze_df)

            result.insert(0, "condition", condition)
            result.insert(0, "seed", seed)
            all_rows.append(result)

            summary = summarize_cloze(result)
            summary_rows.append({"seed": seed, "condition": condition, **summary})

            del model
            cleanup_cuda()

    cloze_results = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    cloze_summary = pd.DataFrame(summary_rows)

    out1 = EVAL_DIR / "cloze_item_candidate_results.csv"
    out2 = EVAL_DIR / "aggregate_cloze_summary.csv"
    cloze_results.to_csv(out1, index=False)
    cloze_summary.to_csv(out2, index=False)
    print("Saved:", out1)
    print("Saved:", out2)
    display(cloze_summary)
else:
    print("Cloze eval disabled. Set CLOZE_CSV_PATH and RUN_CLOZE_EVAL = True.")


Cloze eval disabled. Set CLOZE_CSV_PATH and RUN_CLOZE_EVAL = True.


Try entropy+iv

In [ ]:

# Entropy × Next-token IV diagnostic
# Based on Similarity-adjusted Surprisal / Information Value


@torch.no_grad()
def exact_next_token_iv_sas_for_model(
    model,
    reference_model,
    entropy_stats,
    dataloader,
    max_batches=10,
    chunk_size=32,
    save_token_rows=True,
):
    """
    Computes token-level:
      standard surprisal: -log p_model(y | context)
      next-token IV: sum_v d(y,v) p_model(v | context)
      similarity-adjusted surprisal: -log sum_v z(y,v) p_model(v | context)

    Similarity:
      z(y,v) = 0.5 * (cos(e_y, e_v) + 1)
      d(y,v) = 1 - z(y,v)

    This is exact over the GPT-2 subword vocabulary, but chunked over token positions.
    """

    model.eval()
    reference_model.eval()

    # Use reference embedding space, same as soft target construction.
    E = get_vocab_emb_norm(reference_model).to(device).float()  # [V, D]

    token_rows = []

    for bidx, batch in enumerate(tqdm(dataloader, desc="Entropy × IV diagnostic")):
        if max_batches is not None and bidx >= max_batches:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        # Model distribution p_model(. | context)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1, :].float()
        labels = input_ids[:, 1:].contiguous()
        mask = shift_attention_mask(attention_mask, labels).bool()

        # Reference entropy signal used by dynamic functions
        ref_entropy = compute_reference_entropy(reference_model, input_ids, attention_mask)
        h_norm = normalize_entropy(ref_entropy, entropy_stats)

        B, T, V = logits.shape

        logits_flat = logits.reshape(B * T, V)
        labels_flat = labels.reshape(B * T)
        mask_flat = mask.reshape(B * T)
        ref_entropy_flat = ref_entropy.reshape(B * T)
        h_norm_flat = h_norm.reshape(B * T)

        valid_idx = torch.nonzero(mask_flat, as_tuple=False).squeeze(-1)

        for start in range(0, valid_idx.numel(), chunk_size):
            idx = valid_idx[start : start + chunk_size]

            chunk_logits = logits_flat[idx]       # [n, V]
            y = labels_flat[idx]                  # [n]
            chunk_ref_entropy = ref_entropy_flat[idx]
            chunk_h_norm = h_norm_flat[idx]

            log_probs = F.log_softmax(chunk_logits, dim=-1)
            probs = torch.exp(log_probs)

            # Standard surprisal
            surprisal = -log_probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)

            # z(y,v) = 0.5 * (cos + 1)
            gold_emb = E[y]                       # [n, D]
            cos = gold_emb @ E.T                  # [n, V], cosine because E is normalized
            z = 0.5 * (cos + 1.0)                 # [n, V], in [0,1]

            # similarity mass = sum_v z(y,v) p(v|context)
            sim_mass = (z * probs).sum(dim=-1).clamp_min(1e-12)

            # IV = sum_v (1-z(y,v)) p(v|context) = 1 - sim_mass
            iv = 1.0 - sim_mass

            # Similarity-adjusted surprisal
            sas = -torch.log(sim_mass)

            # Some useful extra diagnostics
            gold_prob = torch.exp(-surprisal)
            top1 = torch.argmax(chunk_logits, dim=-1)
            top1_correct = (top1 == y).float()

            if save_token_rows:
                for j in range(idx.numel()):
                    token_id = int(y[j].detach().cpu().item())
                    token_rows.append({
                        "batch": int(bidx),
                        "token_id": token_id,
                        "token": tokenizer.decode([token_id]),
                        "reference_entropy": float(chunk_ref_entropy[j].detach().cpu().item()),
                        "h_norm": float(chunk_h_norm[j].detach().cpu().item()),
                        "surprisal": float(surprisal[j].detach().cpu().item()),
                        "gold_prob": float(gold_prob[j].detach().cpu().item()),
                        "next_token_iv": float(iv[j].detach().cpu().item()),
                        "similarity_adjusted_surprisal": float(sas[j].detach().cpu().item()),
                        "top1_correct": float(top1_correct[j].detach().cpu().item()),
                    })

    df = pd.DataFrame(token_rows)

    if len(df) == 0:
        return df, pd.DataFrame(), pd.DataFrame()

    # Entropy buckets
    df["entropy_bucket"] = pd.cut(
        df["h_norm"],
        bins=[-1e-9, 0.33, 0.66, 1.0 + 1e-9],
        labels=["low_entropy", "mid_entropy", "high_entropy"],
    )

    # IV buckets by quantiles
    q33 = df["next_token_iv"].quantile(0.33)
    q66 = df["next_token_iv"].quantile(0.66)
    df["iv_bucket"] = pd.cut(
        df["next_token_iv"],
        bins=[-1e-9, q33, q66, df["next_token_iv"].max() + 1e-9],
        labels=["low_iv", "mid_iv", "high_iv"],
    )

    # low/high entropy × low/high IV
    h_med = df["h_norm"].median()
    iv_med = df["next_token_iv"].median()

    df["entropy_highlow"] = np.where(df["h_norm"] >= h_med, "high_entropy", "low_entropy")
    df["iv_highlow"] = np.where(df["next_token_iv"] >= iv_med, "high_iv", "low_iv")
    df["entropy_iv_quadrant"] = df["entropy_highlow"] + "__" + df["iv_highlow"]

    bucket_summary = (
        df.groupby(["entropy_bucket", "iv_bucket"], observed=True)
        .agg(
            n=("token_id", "count"),
            mean_surprisal=("surprisal", "mean"),
            mean_gold_prob=("gold_prob", "mean"),
            mean_iv=("next_token_iv", "mean"),
            mean_sas=("similarity_adjusted_surprisal", "mean"),
            mean_h_norm=("h_norm", "mean"),
            top1_acc=("top1_correct", "mean"),
        )
        .reset_index()
    )

    quadrant_summary = (
        df.groupby("entropy_iv_quadrant")
        .agg(
            n=("token_id", "count"),
            mean_surprisal=("surprisal", "mean"),
            mean_gold_prob=("gold_prob", "mean"),
            mean_iv=("next_token_iv", "mean"),
            mean_sas=("similarity_adjusted_surprisal", "mean"),
            mean_h_norm=("h_norm", "mean"),
            top1_acc=("top1_correct", "mean"),
        )
        .reset_index()
    )

    corr_summary = pd.DataFrame([{
        "corr_hnorm_iv": df["h_norm"].corr(df["next_token_iv"]),
        "corr_hnorm_surprisal": df["h_norm"].corr(df["surprisal"]),
        "corr_hnorm_sas": df["h_norm"].corr(df["similarity_adjusted_surprisal"]),
        "corr_surprisal_iv": df["surprisal"].corr(df["next_token_iv"]),
        "corr_surprisal_sas": df["surprisal"].corr(df["similarity_adjusted_surprisal"]),
        "n_tokens": len(df),
    }])

    return df, bucket_summary, quadrant_summary, corr_summary


def run_entropy_iv_diagnostic_for_conditions(
    seed=42,
    conditions=None,
    max_batches=10,
    chunk_size=32,
):
    if conditions is None:
        conditions = CONDITIONS

    _, valid_loader = make_dataloaders(seed, train_shuffle=False)
    signal_loader, _ = make_dataloaders(seed, train_shuffle=False)

    reference_model, entropy_stats, vocab_emb_norm = get_or_build_reference_signals(seed, signal_loader)

    all_token_rows = []
    all_bucket_rows = []
    all_quadrant_rows = []
    all_corr_rows = []

    for condition in conditions:
        ckpt = selected_checkpoint_dir(seed, condition)
        if ckpt is None:
            print("Missing checkpoint:", condition)
            continue

        print("=" * 80)
        print("Entropy × IV diagnostic:", condition)
        print("=" * 80)

        model = GPT2LMHeadModel.from_pretrained(ckpt).to(device)

        token_df, bucket_df, quadrant_df, corr_df = exact_next_token_iv_sas_for_model(
            model=model,
            reference_model=reference_model,
            entropy_stats=entropy_stats,
            dataloader=valid_loader,
            max_batches=max_batches,
            chunk_size=chunk_size,
            save_token_rows=True,
        )

        token_df.insert(0, "condition", condition)
        token_df.insert(0, "seed", seed)

        bucket_df.insert(0, "condition", condition)
        bucket_df.insert(0, "seed", seed)

        quadrant_df.insert(0, "condition", condition)
        quadrant_df.insert(0, "seed", seed)

        corr_df.insert(0, "condition", condition)
        corr_df.insert(0, "seed", seed)

        all_token_rows.append(token_df)
        all_bucket_rows.append(bucket_df)
        all_quadrant_rows.append(quadrant_df)
        all_corr_rows.append(corr_df)

        del model
        cleanup_cuda()

    token_all = pd.concat(all_token_rows, ignore_index=True) if all_token_rows else pd.DataFrame()
    bucket_all = pd.concat(all_bucket_rows, ignore_index=True) if all_bucket_rows else pd.DataFrame()
    quadrant_all = pd.concat(all_quadrant_rows, ignore_index=True) if all_quadrant_rows else pd.DataFrame()
    corr_all = pd.concat(all_corr_rows, ignore_index=True) if all_corr_rows else pd.DataFrame()

    out_token = DIAG_DIR / f"entropy_next_token_iv_token_level_seed_{seed}.csv"
    out_bucket = DIAG_DIR / f"entropy_next_token_iv_bucket_summary_seed_{seed}.csv"
    out_quad = DIAG_DIR / f"entropy_next_token_iv_quadrant_summary_seed_{seed}.csv"
    out_corr = DIAG_DIR / f"entropy_next_token_iv_correlations_seed_{seed}.csv"

    token_all.to_csv(out_token, index=False)
    bucket_all.to_csv(out_bucket, index=False)
    quadrant_all.to_csv(out_quad, index=False)
    corr_all.to_csv(out_corr, index=False)

    print("Saved:", out_token)
    print("Saved:", out_bucket)
    print("Saved:", out_quad)
    print("Saved:", out_corr)

    display(corr_all)
    display(bucket_all)
    display(quadrant_all)

    del reference_model, vocab_emb_norm
    cleanup_cuda()

    return token_all, bucket_all, quadrant_all, corr_all

In [ ]:
test_conditions = [
    "hard_ce",
    "full_vocab_static_tau_0p055",
    "full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9",
    "full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9_shuffled_tau",
]

token_iv_df, bucket_iv_df, quadrant_iv_df, corr_iv_df = run_entropy_iv_diagnostic_for_conditions(
    seed=42,
    conditions=test_conditions,
    max_batches=10,   # 先小测试，确认能跑；之后可改 50 或 None
    chunk_size=32,
)

Reference checkpoint: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/checkpoints/seed_42/hard_ce/best_by_val_loss
Entropy stats: {'mean': 3.8352062702178955, 'std': 2.2439565658569336, 'q01': 0.0027693677693605423, 'q05': 0.006557294633239508, 'q25': 2.4359638690948486, 'q50': 4.047797679901123, 'q75': 5.367341041564941, 'q95': 7.346242427825928, 'q99': 8.066534042358398, 'n': 16135635}
Entropy × IV diagnostic: hard_ce


Entropy × IV diagnostic:   0%|          | 0/81 [00:00<?, ?it/s]

Entropy × IV diagnostic: full_vocab_static_tau_0p055


Entropy × IV diagnostic:   0%|          | 0/81 [00:00<?, ?it/s]

Entropy × IV diagnostic: full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9


...

Entropy × IV diagnostic:   0%|          | 0/81 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/diagnostics/entropy_next_token_iv_token_level_seed_42.csv
Saved: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/diagnostics/entropy_next_token_iv_bucket_summary_seed_42.csv
Saved: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/diagnostics/entropy_next_token_iv_quadrant_summary_seed_42.csv
Saved: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/diagnostics/entropy_next_token_iv_correlations_seed_42.csv


,seed,condition,corr_hnorm_iv,corr_hnorm_surprisal,corr_hnorm_sas,corr_surprisal_iv,corr_surprisal_sas,n_tokens
0,42,hard_ce,0.843887,0.668647,0.803857,0.828090,0.842433,20400
1,42,full_vocab_static_tau_0p055,0.847282,0.676482,0.808774,0.815912,0.826986,20400
2,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,0.837827,0.666734,0.796916,0.830969,0.844764,20400
3,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,0.837954,0.666295,0.797163,0.831482,0.845465,20400


,seed,condition,entropy_bucket,iv_bucket,n,mean_surprisal,mean_gold_prob,mean_iv,mean_sas,mean_h_norm,top1_acc
0,42,hard_ce,low_entropy,low_iv,4976,0.301807,0.839572,0.027756,0.029155,0.079023,0.905949
1,42,hard_ce,low_entropy,mid_iv,168,3.983016,0.056601,0.257396,0.298963,0.233367,0.011905
2,42,hard_ce,low_entropy,high_iv,131,8.047547,0.009488,0.437434,0.591102,0.250284,0.000000
3,42,hard_ce,mid_entropy,low_iv,1716,1.006069,0.407095,0.155516,0.169464,0.451592,0.885781
4,42,hard_ce,mid_entropy,mid_iv,4363,3.268191,0.095150,0.265813,0.310124,0.532010,0.225762
5,42,hard_ce,mid_entropy,high_iv,1910,6.965367,0.012549,0.407192,0.531601,0.546653,0.009424
6,42,hard_ce,high_entropy,low_iv,40,3.629226,0.105667,0.165420,0.181529,0.753737,0.350000
7,42,hard_ce,high_entropy,mid_iv,2201,4.328785,0.053123,0.286153,0.338053,0.778426,0.218537
8,42,hard_ce,high_entropy,high_iv,4895,6.688745,0.015367,0.409248,0.531310,0.850227,0.065169
9,42,full_vocab_static_tau_0p055,low_entropy,low_iv,4951,0.322716,0.824005,0.031145,0.032810,0.078292,0.903252


,seed,condition,entropy_iv_quadrant,n,mean_surprisal,mean_gold_prob,mean_iv,mean_sas,mean_h_norm,top1_acc
0,42,hard_ce,high_entropy__high_iv,8324,5.970304,0.025369,0.378853,0.482919,0.783539,0.091663
1,42,hard_ce,high_entropy__low_iv,1876,3.030764,0.116661,0.242855,0.278833,0.657558,0.367271
2,42,hard_ce,low_entropy__high_iv,1876,5.946291,0.024582,0.360007,0.455835,0.451915,0.015458
3,42,hard_ce,low_entropy__low_iv,8324,1.008051,0.606244,0.095888,0.106438,0.229120,0.764777
4,42,full_vocab_static_tau_0p055,high_entropy__high_iv,8297,6.119939,0.024001,0.386105,0.494999,0.784877,0.087743
5,42,full_vocab_static_tau_0p055,high_entropy__low_iv,1903,3.051087,0.124843,0.245402,0.282323,0.653508,0.366789
6,42,full_vocab_static_tau_0p055,low_entropy__high_iv,1903,5.880102,0.027665,0.363138,0.460227,0.452150,0.036259
7,42,full_vocab_static_tau_0p055,low_entropy__low_iv,8297,1.040336,0.594064,0.100524,0.111911,0.228341,0.759070
8,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,high_entropy__high_iv,8265,6.004902,0.024557,0.378984,0.483452,0.783728,0.084211
9,42,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,high_entropy__low_iv,1935,2.931304,0.129823,0.238329,0.273015,0.660592,0.389664


In [ ]:
test_conditions = [
    "hard_ce",
    "full_vocab_static_tau_0p055",
    "full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9",
    "full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9_shuffled_tau",
]

token_iv_df, bucket_iv_df, quadrant_iv_df, corr_iv_df = run_entropy_iv_diagnostic_for_conditions(
    seed= 43,
    conditions=test_conditions,
    max_batches=50,   # 先小测试，之后改 50 或 None
    chunk_size=32,
    )

Reference checkpoint: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/checkpoints/seed_43/hard_ce/best_by_val_loss
Entropy stats: {'mean': 3.8217828273773193, 'std': 2.243056297302246, 'q01': 0.0023842905648052692, 'q05': 0.006430407986044884, 'q25': 2.4385576248168945, 'q50': 4.063372611999512, 'q75': 5.3588175773620605, 'q95': 7.31091833114624, 'q99': 8.00882625579834, 'n': 16135635}
Entropy × IV diagnostic: hard_ce


Entropy × IV diagnostic:   0%|          | 0/81 [00:00<?, ?it/s]

Entropy × IV diagnostic: full_vocab_static_tau_0p055


Entropy × IV diagnostic:   0%|          | 0/81 [00:00<?, ?it/s]

Entropy × IV diagnostic: full_vocab_sigmoid_hard_gate_tau_max_0p055_gate_0p5_mix_alpha_0p9


...

Entropy × IV diagnostic:   0%|          | 0/81 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/diagnostics/entropy_next_token_iv_token_level_seed_43.csv
Saved: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/diagnostics/entropy_next_token_iv_bucket_summary_seed_43.csv
Saved: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/diagnostics/entropy_next_token_iv_quadrant_summary_seed_43.csv
Saved: /content/drive/MyDrive/babylm_full_vocab_soft_target_runs/v4_full_vocab_core5_paper_128_warmup1k_fasttrain_fullfinal/diagnostics/entropy_next_token_iv_correlations_seed_43.csv


,seed,condition,corr_hnorm_iv,corr_hnorm_surprisal,corr_hnorm_sas,corr_surprisal_iv,corr_surprisal_sas,n_tokens
0,43,hard_ce,0.837040,0.653085,0.794807,0.826004,0.841133,102000
1,43,full_vocab_static_tau_0p055,0.841862,0.664170,0.801879,0.814495,0.827750,102000
2,43,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,0.833144,0.655894,0.792268,0.822893,0.837618,102000
3,43,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,0.832065,0.654570,0.790490,0.825708,0.840507,102000


,seed,condition,entropy_bucket,iv_bucket,n,mean_surprisal,mean_gold_prob,mean_iv,mean_sas,mean_h_norm,top1_acc
0,43,hard_ce,low_entropy,low_iv,23250,0.310296,0.841365,0.026824,0.028210,0.074673,0.897204
1,43,hard_ce,low_entropy,mid_iv,823,4.098399,0.056885,0.268985,0.314517,0.237102,0.009721
2,43,hard_ce,low_entropy,high_iv,490,8.302970,0.007382,0.440176,0.593953,0.234313,0.000000
3,43,hard_ce,mid_entropy,low_iv,10089,1.159717,0.381670,0.160476,0.175544,0.456045,0.827238
4,43,hard_ce,mid_entropy,mid_iv,21874,3.389966,0.085904,0.268441,0.313676,0.541327,0.201792
5,43,hard_ce,mid_entropy,high_iv,9505,7.182134,0.010219,0.413874,0.543402,0.550517,0.007680
6,43,hard_ce,high_entropy,low_iv,321,3.319593,0.100494,0.177616,0.196241,0.747550,0.358255
7,43,hard_ce,high_entropy,mid_iv,10963,4.219907,0.061088,0.292549,0.346947,0.791971,0.252303
8,43,hard_ce,high_entropy,high_iv,24685,6.715853,0.014648,0.411593,0.535261,0.852227,0.059105
9,43,full_vocab_static_tau_0p055,low_entropy,low_iv,23219,0.335927,0.821511,0.031805,0.033605,0.074370,0.896378


,seed,condition,entropy_iv_quadrant,n,mean_surprisal,mean_gold_prob,mean_iv,mean_sas,mean_h_norm,top1_acc
0,43,hard_ce,high_entropy__high_iv,41395,6.020561,0.025042,0.382016,0.487997,0.791291,0.089262
1,43,hard_ce,high_entropy__low_iv,9605,3.054099,0.111603,0.246189,0.283304,0.670446,0.376158
2,43,hard_ce,low_entropy__high_iv,9605,6.073806,0.022759,0.364809,0.463716,0.462494,0.015096
3,43,hard_ce,low_entropy__low_iv,41395,1.086824,0.584018,0.102130,0.113708,0.242613,0.738930
4,43,full_vocab_static_tau_0p055,high_entropy__high_iv,41244,6.153822,0.023562,0.388766,0.499083,0.792469,0.089298
5,43,full_vocab_static_tau_0p055,high_entropy__low_iv,9756,3.177646,0.113340,0.250501,0.289155,0.667337,0.357421
6,43,full_vocab_static_tau_0p055,low_entropy__high_iv,9756,5.980016,0.025784,0.367429,0.467103,0.463132,0.033108
7,43,full_vocab_static_tau_0p055,low_entropy__low_iv,41244,1.119668,0.570559,0.108065,0.120727,0.241657,0.737271
8,43,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,high_entropy__high_iv,41119,6.042554,0.024214,0.383332,0.490096,0.791137,0.087891
9,43,full_vocab_sigmoid_hard_gate_tau_max_0p055_gat...,high_entropy__low_iv,9881,3.055409,0.115271,0.245220,0.282154,0.674464,0.374254
